# 📄 Scientific Paper RAG Pipeline — Neo4j Edition
### CSAI415 – Special Topics in AI | Hands-On Lab

**Full pipeline:**  
`PDF + .bib pair` → Reference Extraction → Citation Graph → Chunking → Embedding → Qdrant + Neo4j + MongoDB → `/search` API

---

## What changed vs. Week 5

In Week 5 we stored the **citation graph as embedded arrays** inside MongoDB documents (`cites[]`, `cited_by[]`). That worked, but it modelled citations as *fields on a record* rather than as **first-class relationships**.

This week we promote citations to first-class entities by moving the **document registry + citation graph into Neo4j**. The change is surgical:

| Layer | Week 5 | Week 6 |
|---|---|---|
| **Paper metadata + citation graph** | MongoDB (`documents` collection, arrays for edges) | **Neo4j** (`(:Paper)` nodes, `[:CITES]` relationships) |
| **Author model** | Flat list of strings on the document | **Neo4j** `(:Author)` nodes, `[:WROTE]` edges, derived `[:CO_AUTHORED_WITH]` |
| Chunk metadata mirror | MongoDB `chunks` collection | MongoDB `chunks` collection *(unchanged)* |
| Vector index | Qdrant in-memory | Qdrant in-memory *(unchanged)* |
| Everything else | — | *(unchanged)* |

### Why this split?

- **Neo4j** is purpose-built for graph traversal. Citations are relationships — modelling them as `[:CITES {confidence, match_tier}]` edges between `(:Paper)` nodes makes queries like *"papers cited by the papers that cite X"* natural Cypher one-liners.
- **MongoDB** still earns its keep for the **chunks** collection — chunks aren't graph entities, they're documents with flexible metadata that we may want to query in ad-hoc ways.
- **Qdrant** still owns vector search — chunks aren't going anywhere else.

---

## Architecture

```
papers/
  VaswaniSPUJGKP17.pdf   ←─── text extraction
  VaswaniSPUJGKP17.bib   ←─── structured metadata
  devlin2018bert.pdf
  devlin2018bert.bib
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│  INGESTION LAYER                                            │
│                                                             │
│  BibParser ──────────────────────────────────────────────▶ │
│    title, authors, year, venue, doi, abstract, keywords     │
│                                                             │
│  ReferenceExtractor (priority chain)                        │
│    1. Embedded .bib in PDF  (confidence_base = 1.00)        │
│    2. Embedded .bbl in PDF  (confidence_base = 0.95)        │
│    3. Regex on reference section (confidence varies)        │
│                                                             │
│  CitationResolver                                           │
│    Tier 1: DOI exact        → confidence 1.00               │
│    Tier 2: cite_key exact   → confidence 0.95               │
│    Tier 3: title fuzzy      → confidence 0.85 (single hit)  │
│                               confidence 0.50 (multi-hit)   │
│    Tier 4: author + year    → confidence 0.60               │
│    No match → create stub   → confidence 1.00               │
└─────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│  STORAGE LAYER                                              │
│                                                             │
│  Neo4j (Aura cloud)                                         │
│    (:Paper {doc_id, status, title, year, …})                │
│    (:Author {name, name_norm})                              │
│    (:Author)-[:WROTE {order}]->(:Paper)                     │
│    (:Paper)-[:CITES {confidence, match_tier}]->(:Paper)     │
│    (:Author)-[:CO_AUTHORED_WITH {paper_count}]-(:Author)    │
│                                                             │
│  MongoDB (chunks only — mongomock)                          │
│    chunks   {chunk_id, doc_id, page_num, text, …}           │
│                                                             │
│  Qdrant (in-memory)                                         │
│    collection: scientific_papers                            │
│    vector: BGE-small-en-v1.5 (384D, cosine)                 │
└─────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│  RETRIEVAL LAYER                                            │
│                                                             │
│  FastAPI                                                    │
│    GET  /search?q=…&mode=hybrid&min_confidence=0.8          │
│    GET  /documents                                          │
│    GET  /document/{doc_id}/citations                        │
│    GET  /document/{doc_id}/cited_by                         │
│    POST /ingest        (PDF + .bib upload)                  │
│    POST /reconcile     (resolve stubs against new papers)   │
└─────────────────────────────────────────────────────────────┘
```

---

## Sections
1. Environment Setup  
2. BibTeX Parser  
3. Reference Extractor — embedded `.bib` / `.bbl` / regex  
4. Citation Resolver & Confidence Scoring  
5. PDF Text Extraction & Chunking  
6. Embedding with BGE-small  
7. Qdrant Vector Store  
8. **Neo4j — Paper Nodes & `[:CITES]` Relationships** ⭐ *(replaces MongoDB documents)*  
9. Full Ingestion Pipeline  
10. BM25 Index  
11. FastAPI `/search` + Citation Endpoints  
12. End-to-End Demo  
13. Exercises

## Section 1 — Environment Setup

We install everything up front so the rest of the notebook runs without interruption.

| Package | Purpose |
|---|---|
| `pymupdf` | PDF parsing — text, blocks, embedded files |
| `pymupdf4llm` | Optional ML-assisted layout mode |
| `bibtexparser` | Parse `.bib` files into Python dicts |
| `sentence-transformers` | BGE-small embeddings |
| `qdrant-client` | Vector store (in-memory, no Docker) |
| **`neo4j`** | **Official Neo4j Python driver (connects to Aura cloud)** |
| `pymongo` + `mongomock` | MongoDB interface for chunks collection only |
| `rank_bm25` | BM25 sparse retrieval |
| `rapidfuzz` | Fast fuzzy string matching for title resolution |
| `fastapi` + `uvicorn` | REST API |
| `nest-asyncio` | Allows asyncio inside Jupyter |

### Neo4j connection

We connect to a **Neo4j Aura free instance** (cloud-hosted). No Docker, no local install. Sign up at https://neo4j.com/cloud/aura-free/ and fill in your credentials below.

In [ ]:
!pip install -q \
    pymupdf pymupdf4llm \
    bibtexparser \
    sentence-transformers \
    qdrant-client \
    neo4j \
    pymongo mongomock \
    rank_bm25 \
    rapidfuzz \
    fastapi "uvicorn[standard]" \
    httpx nest-asyncio python-multipart \
    tqdm matplotlib

print("✅ All packages installed")

In [ ]:
import os, re, json, hashlib, threading, warnings, urllib.request
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, field, asdict

import fitz                               # PyMuPDF
import bibtexparser
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from IPython.display import Markdown, display

from rapidfuzz import fuzz

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
)

from neo4j import GraphDatabase
import mongomock

from rank_bm25 import BM25Okapi

import nest_asyncio
nest_asyncio.apply()

warnings.filterwarnings("ignore")
print("✅ Imports ready")

### Neo4j Aura credentials

Fill in your Aura connection details below. You can find them in the Aura console after creating a free instance (the credentials file is downloaded once at instance creation time — keep it safe).

```
NEO4J_URI       → neo4j+s://<dbid>.databases.neo4j.io
NEO4J_USER      → neo4j  (default)
NEO4J_PASSWORD  → <generated when you created the instance>
```

> 💡 **Pro tip:** Set these as environment variables rather than hard-coding them in the notebook. The cell below reads from env first, then falls back to the placeholder strings.

In [ ]:
# ── Neo4j connection ─────────────────────────────────────────────────────────
NEO4J_URI      = os.environ.get("NEO4J_URI",      "neo4j+s://<YOUR_DBID>.databases.neo4j.io")
NEO4J_USER     = os.environ.get("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "<YOUR_PASSWORD>")


# Smoke-test the connection before we go further
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    result = session.run("RETURN 'Neo4j connection OK' AS msg").single()
    print(f"✅ {result['msg']}")
driver.close()

## Section 2 — BibTeX Parser

Every paper enters the system as a **pair of files** that share the same filename stem:

```
papers/VaswaniSPUJGKP17.pdf
papers/VaswaniSPUJGKP17.bib
```

The `.bib` file is the **authoritative source of structured metadata** — title, authors, year, venue, DOI, abstract, keywords. We read this *first*, before touching the PDF. If the `.bib` is missing we raise an error immediately rather than silently producing an incomplete record.

### Why not extract metadata from the PDF itself?

PDF metadata (the `DocumentInfo` dictionary) is notoriously unreliable:
- Many PDFs have empty or wrong `Title` / `Author` fields
- Two-column arXiv papers often have the abstract merged with the first body paragraph
- Scanned PDFs have no embedded text metadata at all

The `.bib` file gives us a clean, human-curated record with zero parsing ambiguity.

### The `cite_key` becomes our `doc_id`

BibTeX cite keys (e.g. `VaswaniSPUJGKP17`) are:
- **Human-readable** — immediately meaningful in logs and APIs
- **Unique by convention** — the author assigned it deliberately
- **Stable** — they don't change when the paper is reformatted
- **Already present** in `\cite{}` commands throughout other papers' source

This makes them ideal as the primary key for `(:Paper)` nodes in Neo4j and as vector payload identifiers in Qdrant.

In [ ]:
@dataclass
class DocMetadata:
    '''
    Structured representation of a paper's bibliographic metadata.
    Populated from the .bib file; PDF contributes only chunk text.

    Note: cites/cited_by are NOT stored on this dataclass anymore —
    they live as Neo4j relationships now, not as embedded arrays.
    '''
    doc_id:      str                      # BibTeX cite key
    status:      str       = "ingested"   # "ingested" | "stub"
    title:       str       = ""
    authors:     List[str] = field(default_factory=list)
    year:        Optional[int] = None
    venue:       str       = ""           # journal / conference
    doi:         str       = ""
    abstract:    str       = ""
    keywords:    List[str] = field(default_factory=list)
    source:      str       = ""           # original filename
    ingested_at: str       = ""

    def to_dict(self) -> Dict:
        return asdict(self)


class BibParser:
    '''
    Parses a .bib file and returns a DocMetadata object.

    The BibTeX cite key becomes the doc_id.
    Multi-author fields are split on ' and ' (standard BibTeX convention).
    Keywords are split on ',' or ';'.
    '''

    def parse(self, bib_path: str) -> DocMetadata:
        path = Path(bib_path)
        assert path.exists(), f"Missing .bib file: {bib_path}"

        with open(path, encoding="utf-8", errors="replace") as f:
            raw = f.read()

        # bibtexparser v1 API
        parser = bibtexparser.bparser.BibTexParser(common_strings=True)
        db     = bibtexparser.loads(raw, parser=parser)

        if not db.entries:
            raise ValueError(f"No BibTeX entries found in {bib_path}")

        # Use the first entry (a paper's .bib should have exactly one)
        entry = db.entries[0]

        authors = self._parse_authors(entry.get("author", ""))
        keywords = self._parse_keywords(entry.get("keywords", ""))

        year_raw = entry.get("year", "")
        try:
            year = int(year_raw)
        except (ValueError, TypeError):
            year = None

        venue = (
            entry.get("journal")
            or entry.get("booktitle")
            or entry.get("publisher")
            or ""
        )

        return DocMetadata(
            doc_id      = entry.get("ID", path.stem),
            title       = self._clean(entry.get("title", path.stem)),
            authors     = authors,
            year        = year,
            venue       = self._clean(venue),
            doi         = entry.get("doi", "").strip(),
            abstract    = self._clean(entry.get("abstract", "")),
            keywords    = keywords,
            source      = path.stem,
            ingested_at = datetime.utcnow().isoformat(),
        )

    def _parse_authors(self, raw: str) -> List[str]:
        '''Split 'Last, First and Last2, First2' into a list.'''
        if not raw.strip():
            return []
        parts = re.split(r"\s+and\s+", raw, flags=re.IGNORECASE)
        return [p.strip() for p in parts if p.strip()]

    def _parse_keywords(self, raw: str) -> List[str]:
        if not raw.strip():
            return []
        return [k.strip() for k in re.split(r"[,;]", raw) if k.strip()]

    def _clean(self, text: str) -> str:
        '''Remove LaTeX braces and normalise whitespace.'''
        text = re.sub(r"\{([^}]*)\}", r"\1", text)   # {word} → word
        text = re.sub(r"\s+", " ", text)
        return text.strip()


# ── Demo ─────────────────────────────────────────────────────────────────────
# We'll download the Attention Is All You Need paper from arXiv for testing.
# Its .bib is provided inline below.

SAMPLE_BIB = '''
@inproceedings{vaswani2017attention,
  author    = {Ashish Vaswani and Noam Shazeer and Niki Parmar and
               Jakob Uszkoreit and Llion Jones and Aidan N. Gomez and
               Lukasz Kaiser and Illia Polosukhin},
  title     = {Attention Is All You Need},
  booktitle = {Advances in Neural Information Processing Systems},
  year      = {2017},
  doi       = {10.48550/arXiv.1706.03762},
  keywords  = {transformer, attention, neural machine translation},
  abstract  = {The dominant sequence transduction models are based on complex
               recurrent or convolutional neural networks. We propose a new
               simple network architecture, the Transformer, based solely on
               attention mechanisms.}
}
'''

# Write demo .bib file
# Path("vaswani2017attention.bib").write_text(SAMPLE_BIB)

bib_parser = BibParser()
meta = bib_parser.parse("./papers/VaswaniSPUJGKP17.bib")

print("📋 Parsed metadata:")
for k, v in meta.to_dict().items():
    print(f"  {k:15s}: {v}")
meta2 = bib_parser.parse("./papers/devlin2018bert.bib")      

print("📋 Parsed metadata:")
for k, v in meta.to_dict().items():
    print(f"  {k:15s}: {v}")

## Section 3 — Reference Extractor

Extracting the list of references a paper cites is harder than it looks. The challenge is that the PDF renders references as *visual text* — we need to reverse-engineer the structure from what was originally machine-readable (LaTeX source).

We use a **three-tier priority chain**, trying the most reliable source first:

---

### Priority 1 — Embedded `.bib` file (confidence base: 1.0)

Some authors and publishers attach the original BibTeX file as an embedded stream inside the PDF. This is increasingly common with arXiv submissions. If present, it's the ground truth — structured, cite-keyed, complete.

```python
doc.embfile_count()          # how many embedded files
doc.embfile_info(i)          # name, size, etc.
doc.embfile_get(i)           # raw bytes
```

---

### Priority 2 — Embedded `.bbl` file (confidence base: 0.95)

The `.bbl` file is the *compiled* bibliography that LaTeX generates from the `.bib` + BibTeX style. It looks like:

```latex
\bibitem[Vaswani et al.(2017)]{vaswani2017attention}
  Vaswani, A., et al.
  \newblock Attention is all you need.
  \newblock {\em NeurIPS}, 2017.
```

The cite key `vaswani2017attention` is explicitly present — that's the most valuable piece for matching. We don't need to parse the full author/title fields from the `.bbl` since we can use the cite key to look up the record in Neo4j.

---

### Priority 3 — Regex on the rendered reference section (confidence: varies)

When nothing is embedded, we fall back to parsing the rendered text of the reference section. The strategy:

1. **Locate** the reference section — find the last page containing a heading that matches `References`, `Bibliography`, or `Works Cited`
2. **Segment** individual entries — split on `[N]` numbered markers, or on hanging-indent patterns
3. **Parse** each entry with regex patterns for author/year/title/venue

This is inherently fuzzy. The confidence score assigned downstream reflects the quality of the match against existing Neo4j records.

---

### The `RawReference` dataclass

Regardless of which tier produces the reference, we normalise everything into the same `RawReference` object before passing it to the `CitationResolver`.

In [ ]:
@dataclass
class RawReference:
    '''
    A reference entry as extracted from a PDF, before Neo4j resolution.
    The extraction_source tells us which tier produced this entry.
    '''
    raw_text:         str  = ""          # original string from PDF
    cite_key:         Optional[str] = None   # from .bib/.bbl if available
    title:            str  = ""
    authors:          List[str] = field(default_factory=list)
    year:             Optional[int] = None
    doi:              str  = ""
    venue:            str  = ""
    extraction_source: str = "regex"     # "embedded_bib"|"embedded_bbl"|"regex"


class ReferenceExtractor:
    '''
    Extracts the reference list from a PDF using a three-tier priority chain.

    Priority order (most to least reliable):
        1. Embedded .bib file inside the PDF
        2. Embedded .bbl file inside the PDF
        3. Regex parsing of the rendered reference section text

    All three paths return List[RawReference] with a consistent schema.
    The extraction_source field on each entry tells downstream code
    how much to trust the structured fields.
    '''

    # ── Regex patterns for rendered reference text ────────────────────────────

    # Matches [1], [12], [1, 2], [Vas17] style numeric/alphanumeric markers
    RE_NUMBERED = re.compile(r"^\s*\[[\w,\s]+\]\s*", re.MULTILINE)

    # Matches "Author et al. (2017)" or "Author, A. B. (2017)"
    RE_YEAR_PAREN = re.compile(r"\((\d{4}[a-z]?)\)")

    # Matches year written towards the end without parenthesis
    RE_YEAR_NO_PAREN = re.compile(r", (\d{4}).$")
    # DOI patterns
    RE_DOI = re.compile(
        r"(?:doi:|https?://doi\.org/)([^\s,;\]]+)", re.IGNORECASE
    )
    RE_AUTHORS = re.compile(r"^(.+?)\.")

    # Title heuristic: text after year, before venue marker
    # Looks for quoted titles or capitalised phrases
    RE_TITLE_QUOTED = re.compile(r'"([^"]{10,200})"')
    RE_TITLE_PLAIN  = re.compile(
        r"(?:[\.\)]\s)([A-Z][^.]{15,180})\.",
    )

    # Reference section heading
    RE_REF_HEADING = re.compile(
        r"^\s*(References|Bibliography|Works\s+Cited)\s*$",
        re.IGNORECASE | re.MULTILINE,
    )

    def extract(self, pdf_path: str) -> Tuple[List[RawReference], str]:
        '''
        Main entry point. Returns (references, source_tier_used).
        '''
        doc = fitz.open(pdf_path)

        # ── Tier 1: embedded .bib ─────────────────────────────────────────────
        refs = self._try_embedded_bib(doc)
        if refs:
            doc.close()
            return refs, "embedded_bib"

        # ── Tier 2: embedded .bbl ─────────────────────────────────────────────
        refs = self._try_embedded_bbl(doc)
        if refs:
            doc.close()
            return refs, "embedded_bbl"

        # ── Tier 3: regex on rendered text ────────────────────────────────────
        refs = self._parse_rendered_section(doc)
        doc.close()
        return refs, "regex"

    # ── Tier 1 ────────────────────────────────────────────────────────────────

    def _try_embedded_bib(self, doc) -> List[RawReference]:
        '''Look for an embedded .bib file and parse it with bibtexparser.'''
        for i in range(doc.embfile_count()):
            info = doc.embfile_info(i)
            name = info.get("filename", "").lower()
            if name.endswith(".bib"):
                raw_bytes = doc.embfile_get(i)
                try:
                    raw_text = raw_bytes.decode("utf-8", errors="replace")
                    db = bibtexparser.loads(raw_text)
                    refs = []
                    for entry in db.entries:
                        refs.append(RawReference(
                            raw_text  = str(entry),
                            cite_key  = entry.get("ID"),
                            title     = self._clean_latex(entry.get("title", "")),
                            authors   = self._split_authors(entry.get("author", "")),
                            year      = self._parse_year(entry.get("year", "")),
                            doi       = entry.get("doi", "").strip(),
                            venue     = entry.get("journal") or entry.get("booktitle", ""),
                            extraction_source = "embedded_bib",
                        ))
                    if refs:
                        print(f"  📎 Found embedded .bib with {len(refs)} entries")
                        return refs
                except Exception as e:
                    print(f"  ⚠️  Embedded .bib parse error: {e}")
        print("Could not find embedded .bib file")
        return []

    # ── Tier 2 ────────────────────────────────────────────────────────────────

    def _try_embedded_bbl(self, doc) -> List[RawReference]:
        '''Look for an embedded .bbl file and extract cite keys + titles.'''
        for i in range(doc.embfile_count()):
            info = doc.embfile_info(i)
            name = info.get("filename", "").lower()
            if name.endswith(".bbl"):
                raw_bytes = doc.embfile_get(i)
                try:
                    raw_text = raw_bytes.decode("utf-8", errors="replace")
                    return self._parse_bbl(raw_text)
                except Exception as e:
                    print(f"  ⚠️  Embedded .bbl parse error: {e}")
        return []

    def _parse_bbl(self, bbl_text: str) -> List[RawReference]:
        '''
        Parse LaTeX .bbl format.

        Typical structure:
            \\bibitem[label]{cite_key}
              Author, A. (year).
              \\newblock Title of the paper.
              \\newblock {\\em Venue}, pages.
        '''
        refs = []
        # Split on \bibitem
        entries = re.split(r"\\bibitem", bbl_text)[1:]
        for entry in entries:
            cite_key_match = re.match(r"(?:\[[^\]]*\])?\{([^}]+)\}", entry)
            cite_key = cite_key_match.group(1) if cite_key_match else None

            # Title is usually after the first \newblock
            newblocks = re.findall(r"\\newblock\s+([^\n\\]+)", entry)
            title = self._clean_latex(newblocks[0]) if newblocks else ""

            # Year
            year_match = RE_YEAR_PAREN_LOCAL = re.search(r"\((\d{4})\)", entry)
            year = int(year_match.group(1)) if year_match else None

            if cite_key or title:
                refs.append(RawReference(
                    raw_text  = entry[:200],
                    cite_key  = cite_key,
                    title     = title,
                    year      = year,
                    extraction_source = "embedded_bbl",
                ))

        if refs:
            print(f"  📎 Found embedded .bbl with {len(refs)} entries")
        else:
            print("Could not find embedded .bbl file")
        return refs

    # ── Tier 3 ────────────────────────────────────────────────────────────────

    def _parse_rendered_section(self, doc) -> List[RawReference]:
        '''
        Locate the reference section in the rendered PDF text and
        parse individual entries using regex.

        This is the least reliable tier — hence confidence scores
        assigned by CitationResolver will reflect match quality.
        '''
        ref_text = self._extract_reference_section(doc)
        if not ref_text:
            print("  ⚠️  No reference section detected")
            return []
        # print(ref_text)
        entries = self._segment_entries(ref_text)
        print(f"  📝 Regex extracted {len(entries)} reference entries")
        # for entry in entries:
        #     print(entry)
        #     print("*"*50)
        #     print("\n")
        refs = []
        for raw in entries:
            refs.append(self._parse_entry(raw))
        return [r for r in refs if r.title or r.authors]

    def _extract_reference_section(self, doc) -> str:
        '''
        Find the last occurrence of a 'References' heading and
        return all text from that point to end of document.
        '''
        full_text = ""
        ref_start = -1

        for page_num in range(len(doc)):
            page_text = doc[page_num].get_text("text")
            match = list(self.RE_REF_HEADING.finditer(page_text))
            if match:
                # Take the last match on this page
                ref_start = len(full_text) + match[-1].start()
            full_text += page_text + "\n"

        if ref_start == -1:
            return ""
        return full_text[ref_start:]

    def _segment_entries(self, ref_text: str) -> List[str]:
        '''
        Split reference section into individual entries.
        Handles [1] … [N] numbered lists and author-year styles.
        '''
        # Try [N] numbered style first
        parts = self.RE_NUMBERED.split(ref_text)
        parts = [p.strip() for p in parts if len(p.strip()) > 20]
        if len(parts) > 2:
            return parts

        # Fallback: split on blank lines
        parts = re.split(r"\n{2,}", ref_text)
        return [p.strip() for p in parts if len(p.strip()) > 20]

    def _parse_entry(self, raw: str) -> RawReference:
        '''Extract structured fields from a single reference string.'''
        raw_clean = re.sub(r"\s+", " ", raw).strip()
        # print(raw_clean)
        # DOI
        doi_match = self.RE_DOI.search(raw_clean)
        doi = doi_match.group(1) if doi_match else ""

        # Year
        year_match = self.RE_YEAR_PAREN.search(raw_clean)
        if not year_match: #let look for the style where year is at the end of the reference
            year_match = self.RE_YEAR_NO_PAREN.search(raw_clean)

       
        # print(raw)   
        year = int(year_match.group(1)[:-1] if len(year_match.group(1))> 4 else year_match.group(1)) if year_match else 1000
        print("*"*10 , year)
       
            

        # Title — try quoted first, then heuristic capitalised phrase
        title = ""
        tq = self.RE_TITLE_QUOTED.search(raw_clean)
        if tq:
            title = tq.group(1)
        else:
            tp = self.RE_TITLE_PLAIN.search(raw_clean)
            if tp:
                title = tp.group(1)

        # Authors — text before year or title
        authors = []

        author_text = self.RE_AUTHORS.search(raw_clean).group(1)
        # print(author_text)
        authors = self._split_authors(author_text)

        return RawReference(
            raw_text  = raw_clean[:300],
            title     = self._clean_latex(title),
            authors   = authors[:5],     # cap at 5 for storage
            year      = year,
            doi       = doi,
            extraction_source = "regex",
        )

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _clean_latex(self, text: str) -> str:
        text = re.sub(r"\{([^}]*)\}", r"\1", text)
        text = re.sub(r"\\[a-zA-Z]+\s*", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    def _split_authors(self, raw: str) -> List[str]:
        if not raw.strip():
            return []
        parts = re.split(r",\s+and\s+|,\s+(?=[A-Z])", raw, flags=re.IGNORECASE)
        return [p.strip() for p in parts if len(p.strip()) > 1][:10]

    def _parse_year(self, raw: str) -> Optional[int]:
        try:
            return int(str(raw).strip())
        except (ValueError, TypeError):
            return None


# ── Demo ─────────────────────────────────────────────────────────────────────

#PDF_PATH = "vaswani2017attention.pdf"
PDF_PATH = "./papers/VaswaniSPUJGKP17.pdf"
if not Path(PDF_PATH).exists():
    print("Downloading sample PDF...")
    urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", PDF_PATH)
    print("✅ Downloaded")

extractor = ReferenceExtractor()
raw_refs, source_tier = extractor.extract(PDF_PATH)

print(f"\n📚 Extraction tier used : {source_tier}")
print(f"   References found     : {len(raw_refs)}")
print(f"\n🔍 First 3 references:")
for r in raw_refs[:3]:
    print(f"\n  cite_key : {r.cite_key}")
    print(f"  title    : {r.title[:80]}")
    print(f"  authors  : {r.authors[:3]}")
    print(f"  year     : {r.year}")
    print(f"  doi      : {r.doi}")
    print(f"  source   : {r.extraction_source}")

## Section 4 — Citation Resolver & Confidence Scoring

The `CitationResolver` takes each `RawReference` and tries to find its corresponding `(:Paper)` node in Neo4j. This is a **matching problem** — the reference string in paper B may describe paper A in a different format than how A is stored.

### The Four-Tier Matching Strategy

We try tiers in order, stopping at the first successful match:

```
Tier 1 — DOI exact match           confidence = 1.00
          Most reliable. DOIs are globally unique identifiers.
          Fails when the reference doesn't include a DOI (common in older papers).

Tier 2 — cite_key exact match      confidence = 0.95  
          Only available when we got the reference from an embedded .bib/.bbl.
          The same cite key in two papers is almost certainly the same paper.

Tier 3 — Title fuzzy match         confidence = 0.85 (unique hit)
                                   confidence = 0.50 (multiple hits)
          We normalise both titles (lowercase, strip punctuation) then use
          RapidFuzz's token_sort_ratio to handle word-order differences.
          If TWO OR MORE papers in Neo4j exceed the similarity threshold,
          we flag it as ambiguous (confidence 0.50) and pick the best match.

Tier 4 — First-author surname + year  confidence = 0.60
          Last resort. "Vaswani 2017" is usually enough to identify a paper
          within a subdomain, but two papers by the same first author in the
          same year would create a false match.
```

### What happens when no tier matches?

A **stub paper node** is created in Neo4j for the cited paper:
- `status = "stub"` — signals that no PDF has been ingested yet
- A `[:CITES]` edge from the citing paper is created immediately
- `doc_id` is derived from the cite key if available, otherwise slugified from `firstauthor_year_firstword`

When the stub paper is later ingested (PDF + .bib pair arrives), the ingestion pipeline calls `upgrade_stub()` which:
1. Finds the stub `(:Paper {doc_id})` by `doc_id`
2. Fills in all metadata properties from the `.bib`
3. Sets `status = "ingested"`
4. **Preserves all existing incoming `[:CITES]` relationships** — that's the whole point of stubs

### The confidence score lives on the edge

This is where Neo4j shines compared to MongoDB. In Week 5 we stored:

```json
"cites": [
  {"doc_id": "bahdanau2014neural", "confidence": 1.00, "match_tier": "doi"}
]
```

— a JSON array on the document. In Week 6 the same information becomes a **first-class relationship with properties**:

```cypher
(:Paper {doc_id: "vaswani2017attention"})
  -[:CITES {confidence: 1.00, match_tier: "doi", ref_title: "..."}]->
(:Paper {doc_id: "bahdanau2014neural"})
```

The score is on the **edge** (the citation relationship), not the node (the paper). A paper's existence is certain; our confidence that *this reference points to that paper* is what varies. This is the correct modelling — and now it's expressed directly in the data model rather than buried in a sub-document.

In [ ]:
class CitationResolver:
    '''
    Matches RawReference objects against existing Neo4j (:Paper) nodes.

    Matching tiers (highest confidence first):
        1. DOI exact match          → confidence 1.00
        2. cite_key exact match     → confidence 0.95
        3. Title fuzzy match        → confidence 0.85 (unique) / 0.50 (ambiguous)
        4. First-author + year      → confidence 0.60
        No match → create stub      → confidence 1.00

    Each resolved reference becomes a [:CITES] relationship:
        (:Paper)-[:CITES {confidence, match_tier, ref_title}]->(:Paper)
    '''

    FUZZY_THRESHOLD = 88        # RapidFuzz score 0-100
    FUZZY_AMBIGUOUS = 2         # ≥ this many hits → ambiguous (conf 0.50)

    def __init__(self, metadata_store):
        '''
        metadata_store: a Neo4jMetadataStore instance.
        We talk to Neo4j only through that store — no raw driver use here.
        '''
        self.store = metadata_store

    def resolve_all(
        self,
        raw_refs: List[RawReference],
        citing_doc_id: str,
    ) -> List[Dict]:
        '''
        Resolve a list of RawReferences for a given citing paper.
        Returns citation edge dicts and writes [:CITES] relationships to Neo4j.
        '''
        edges = []
        for ref in raw_refs:
            edge = self._resolve_one(ref, citing_doc_id)
            if edge:
                edges.append(edge)
        return edges

    def _resolve_one(
        self,
        ref: RawReference,
        citing_doc_id: str,
    ) -> Optional[Dict]:
        '''Resolve a single reference. Returns a citation edge dict.'''

        matched_doc_id, confidence, tier = None, 0.0, ""

        # ── Tier 1: DOI ───────────────────────────────────────────────────────
        if ref.doi:
            doc = self.store.find_by_doi(ref.doi)
            if doc:
                matched_doc_id = doc["doc_id"]
                confidence, tier = 1.00, "doi"

        # ── Tier 2: cite_key ──────────────────────────────────────────────────
        if not matched_doc_id and ref.cite_key:
            doc = self.store.get_document(ref.cite_key)
            if doc:
                matched_doc_id = doc["doc_id"]
                confidence, tier = 0.95, "cite_key"

        # ── Tier 3: fuzzy title ───────────────────────────────────────────────
        if not matched_doc_id and ref.title:
            matched_doc_id, confidence, tier = self._fuzzy_title_match(ref.title)

        # ── Tier 4: first-author + year ───────────────────────────────────────
        if not matched_doc_id and ref.authors and ref.year:
            matched_doc_id, confidence, tier = self._author_year_match(
                ref.authors[0], ref.year
            )

        # ── No match → create stub ────────────────────────────────────────────
        if not matched_doc_id:
            matched_doc_id = self._create_stub(ref)
            confidence, tier = 1.00, "stub_created"

        # ── Write the [:CITES] edge to Neo4j ──────────────────────────────────
        # Same call for resolved matches and stubs — relationship semantics
        # don't care whether the target was matched or freshly created.
        self.store.create_cites_edge(
            citing_doc_id = citing_doc_id,
            cited_doc_id  = matched_doc_id,
            confidence    = round(confidence, 2),
            match_tier    = tier,
            ref_title     = ref.title[:120],
        )

        return {
            "doc_id":      matched_doc_id,
            "confidence":  round(confidence, 2),
            "match_tier":  tier,
            "ref_title":   ref.title[:120],   # store for debugging
        }

    # ── Matching helpers ──────────────────────────────────────────────────────

    def _fuzzy_title_match(
        self, query_title: str
    ) -> Tuple[Optional[str], float, str]:
        '''
        Normalise both titles and compare with RapidFuzz token_sort_ratio.
        token_sort_ratio handles word-order differences gracefully
        (e.g. "Neural MT by Jointly Learning..." vs "Jointly Learning to Align...").
        '''
        q_norm = self._norm_title(query_title)
        candidates = []

        # Pull all (doc_id, title) pairs from Neo4j and fuzzy-match in Python.
        # For a large corpus you'd use Neo4j full-text indexes instead;
        # at lab scale, fetching all titles is fine.
        for doc in self.store.iter_titles():
            db_norm = self._norm_title(doc.get("title", ""))
            if not db_norm:
                continue
            score = fuzz.token_sort_ratio(q_norm, db_norm)
            if score >= self.FUZZY_THRESHOLD:
                candidates.append((doc["doc_id"], score))

        if not candidates:
            return None, 0.0, ""

        candidates.sort(key=lambda x: x[1], reverse=True)

        if len(candidates) >= self.FUZZY_AMBIGUOUS:
            # Ambiguous — pick best but flag low confidence
            return candidates[0][0], 0.50, "title_fuzzy_ambiguous"

        return candidates[0][0], 0.85, "title_fuzzy"

    def _author_year_match(
        self, first_author: str, year: int
    ) -> Tuple[Optional[str], float, str]:
        '''
        Match on normalised first-author surname + year.
        Extracts surname as last whitespace-separated token before any comma.
        '''
        print("+"*50)
        print(first_author)
        print("+"*50)
        if ("," in first_author):
            surname = first_author.split(",")[0].strip().split()[-1].lower()
        else:
            surname =  first_author.split(" ")[1].strip()
        docs = self.store.find_by_year(year)

        for doc in docs:
            for author in doc.get("authors", []):
                auth_surname = author.split(",")[0].strip().split()[-1].lower()
                if auth_surname == surname:
                    return doc["doc_id"], 0.60, "author_year"

        return None, 0.0, ""

    def _create_stub(self, ref: RawReference) -> str:
        '''
        Create a minimal stub (:Paper) node for an unresolved reference.
        The stub is a real Neo4j node with status='stub'.

        Note: we no longer pass citing_doc_id here — the caller writes
        the [:CITES] edge unconditionally after this returns.
        '''
        # Derive a doc_id: prefer cite_key, then slugify
        if ref.cite_key:
            stub_id = ref.cite_key
        elif ref.authors and ref.year and ref.title:
            surname  = ref.authors[0].split(",")[0].strip().split()[-1].lower()
            word1    = re.sub(r"[^a-z]", "", ref.title.split()[0].lower()) if ref.title else "unknown"
            stub_id  = f"{surname}{ref.year}{word1}"
        else:
            stub_id = f"stub_{hashlib.md5(ref.raw_text.encode()).hexdigest()[:8]}"

        self.store.upsert_stub(
            doc_id  = stub_id,
            title   = ref.title,
            authors = ref.authors,
            year    = ref.year,
            doi     = ref.doi,
            venue   = ref.venue,
        )
        return stub_id

    def _norm_title(self, title: str) -> str:
        '''Lowercase, remove punctuation, collapse whitespace.'''
        t = title.lower()
        t = re.sub(r"[^a-z0-9\s]", " ", t)
        return re.sub(r"\s+", " ", t).strip()

    def upgrade_stub(self, doc_id: str, full_meta: DocMetadata):
        '''
        Upgrade a stub to a full ingested record when the PDF+.bib arrive.

        With Neo4j this is much simpler than the MongoDB version:
        - Incoming [:CITES] relationships are part of the graph, not embedded
          in the document, so they're preserved automatically.
        - We just update the (:Paper) node's properties and flip status.
        '''
        self.store.upgrade_to_ingested(doc_id, full_meta)


print("✅ CitationResolver defined")

## Section 5 — PDF Text Extraction & Chunking

With metadata handled by the `.bib` file and references handled by `ReferenceExtractor`, the PDF's only job is now to provide **body text for chunking and embedding**. This separation of concerns makes each component simpler and more testable.

### Text Extraction

We use PyMuPDF's `get_text("text")` as the default. For most arXiv papers this produces clean, column-ordered text. If you encounter garbled two-column output (words from both columns interleaved), switch to `layout_mode="blocks"` which sorts text blocks left-column-first.

> 💡 **Pro tip:** Run the Layout Sensitivity check in Exercise 1 before processing a new paper corpus — it takes 5 seconds and will tell you if column detection is needed.

### Chunking Strategy

We use a **sliding window** over words (not tokens) because:
- Word counts are language-model agnostic
- 400 words ≈ 512 tokens for English text (close enough for BGE-small's 512-token limit)
- The 50-word overlap preserves context across chunk boundaries — a sentence that starts at the end of chunk N is fully present in chunk N+1

Each chunk inherits rich metadata from the page:
- `doc_id`, `source` — which paper
- `page_num`, `chunk_seq` — position within the paper
- `section` — closest detected heading (useful for filtered search)

### Abstract as a Special Chunk

The abstract from the `.bib` file is stored as a chunk with `chunk_type="abstract"`. This is valuable because:
- Abstracts compress the paper's core contribution into ~150 words — semantically very dense
- Queries like *"papers about positional encoding"* will match abstract chunks first
- We tag it so you can optionally up-weight or filter abstract chunks in search

In [ ]:
class PDFTextExtractor:
    '''
    Extracts body text from a PDF for chunking.
    Metadata comes from the .bib — this class is text-only.

    layout_mode:
        'text'   — fast naive reading order (default, works for most arXiv PDFs)
        'blocks' — column-aware: sorts text blocks left-col first (use for
                   two-column IEEE/ACM papers if 'text' produces garbled output)
    '''

    def __init__(self, layout_mode: str = "text", min_chars: int = 50):
        assert layout_mode in ("text", "blocks")
        self.layout_mode = layout_mode
        self.min_chars   = min_chars

    def extract_pages(self, pdf_path: str, doc_id: str) -> List[Dict]:
        '''
        Returns a list of page dicts:
            {doc_id, source, page_num, text, headings, layout_mode}
        '''
        doc    = fitz.open(pdf_path)
        source = Path(pdf_path).name
        pages  = []

        for page_num, page in enumerate(doc, start=1):
            if self.layout_mode == "blocks":
                text = self._extract_blocks(page)
            else:
                text = page.get_text("text")

            text = self._clean(text)
            if len(text) < self.min_chars:
                continue

            pages.append({
                "doc_id":      doc_id,
                "source":      source,
                "page_num":    page_num,
                "text":        text,
                "headings":    self._extract_headings(page),
                "layout_mode": self.layout_mode,
            })

        doc.close()
        return pages

    def _extract_blocks(self, page) -> str:
        '''
        Column-aware extraction: split page at midpoint, sort each column
        top-to-bottom, then concatenate left column before right column.
        '''
        blocks    = page.get_text("blocks")          # (x0,y0,x1,y1,text,…)
        mid       = page.rect.width / 2
        left_col  = sorted([b for b in blocks if b[0] < mid],  key=lambda b: b[1])
        right_col = sorted([b for b in blocks if b[0] >= mid], key=lambda b: b[1])
        return " ".join(b[4] for b in left_col + right_col if b[4].strip())

    def _extract_headings(self, page) -> List[str]:
        '''Detect bold or large-font spans as section headings.'''
        headings = []
        for block in page.get_text("dict")["blocks"]:
            if block.get("type") != 0:
                continue
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    flags = span.get("flags", 0)
                    size  = span.get("size", 0)
                    text  = span.get("text", "").strip()
                    if text and (bool(flags & 16) or size > 13) and len(text) < 120:
                        headings.append(text)
        return list(dict.fromkeys(headings))

    def _clean(self, text: str) -> str:
        text = re.sub(r"-\n", "", text)          # dehyphenate
        text = re.sub(r"\n+", " ", text)
        text = re.sub(r"[ \t]{2,}", " ", text)
        return text.strip()


class SlidingWindowChunker:
    '''
    Splits page text into overlapping chunks.

    chunk_size : target size in words (≈ tokens for English)
    overlap    : word overlap between consecutive chunks
    '''

    def __init__(self, chunk_size: int = 400, overlap: int = 50):
        self.chunk_size = chunk_size
        self.overlap    = overlap

    def chunk_pages(self, pages: List[Dict], abstract: str = "") -> List[Dict]:
        '''
        Produces chunks from all pages plus an optional abstract chunk.
        The abstract chunk (from .bib) gets chunk_type='abstract'.
        '''
        chunks = []

        # ── Abstract as a special chunk ───────────────────────────────────────
        if abstract and abstract.strip():
            chunks.append({
                "chunk_id":   f"{pages[0]['doc_id']}_abstract",
                "doc_id":     pages[0]["doc_id"],
                "source":     pages[0]["source"],
                "page_num":   0,
                "chunk_seq":  0,
                "section":    "Abstract",
                "chunk_type": "abstract",
                "text":       abstract.strip(),
                "word_count": len(abstract.split()),
            })

        # ── Body text chunks ──────────────────────────────────────────────────
        for page in pages:
            for chunk in self._chunk_text(page):
                chunks.append(chunk)

        return chunks

    def _chunk_text(self, page: Dict) -> List[Dict]:
        words  = page["text"].split()
        if not words:
            return []

        step    = self.chunk_size - self.overlap
        chunks  = []
        idx     = 0
        seq     = 0

        while idx < len(words):
            window = words[idx : idx + self.chunk_size]
            section = page["headings"][0] if page["headings"] else "Body"

            chunks.append({
                "chunk_id":   f"{page['doc_id']}_p{page['page_num']}_c{seq}",
                "doc_id":     page["doc_id"],
                "source":     page["source"],
                "page_num":   page["page_num"],
                "chunk_seq":  seq,
                "section":    section,
                "chunk_type": "body",
                "text":       " ".join(window),
                "word_count": len(window),
            })

            idx += step
            seq += 1

        return chunks


# ── Demo ─────────────────────────────────────────────────────────────────────
text_extractor = PDFTextExtractor(layout_mode="text")
pages          = text_extractor.extract_pages(PDF_PATH, doc_id=meta.doc_id)

chunker = SlidingWindowChunker(chunk_size=400, overlap=50)
chunks  = chunker.chunk_pages(pages, abstract=meta.abstract)

abstract_chunks = [c for c in chunks if c["chunk_type"] == "abstract"]
body_chunks     = [c for c in chunks if c["chunk_type"] == "body"]

print(f"📄 Pages extracted : {len(pages)}")
print(f"🧩 Total chunks    : {len(chunks)}")
print(f"   Abstract chunks : {len(abstract_chunks)}")
print(f"   Body chunks     : {len(body_chunks)}")
print(f"\n📋 Abstract chunk preview:")
print(f"   {chunks[0]['text'][:200]}...")

# Chunk size distribution
word_counts = [c["word_count"] for c in chunks]
plt.figure(figsize=(8, 3))
plt.hist(word_counts, bins=20, color="steelblue", edgecolor="white")
plt.axvline(np.mean(word_counts), color="orange", linestyle="--",
            label=f"Mean: {np.mean(word_counts):.0f} words")
plt.xlabel("Words per Chunk"); plt.ylabel("Frequency")
plt.title("Chunk Size Distribution"); plt.legend(); plt.tight_layout(); plt.show()

## Section 6 — Embedding with BGE-small

We convert each chunk's text into a dense vector using `BAAI/bge-small-en-v1.5`.

### Why BGE-small over MiniLM?

Both are small, fast models suitable for a lab environment. BGE-small wins on MTEB retrieval benchmarks because it was trained with a **query instruction prefix** that explicitly separates the roles of query and document:

- **Documents** are encoded as-is
- **Queries** are prefixed with `"Represent this sentence for searching relevant passages: "`

This asymmetry dramatically improves retrieval precision compared to symmetric encoders like MiniLM.

### Normalisation

We set `normalize_embeddings=True` so all vectors lie on the unit hypersphere. This means **cosine similarity = dot product** — making Qdrant search faster and avoiding the need to configure distance metrics carefully.

### Abstract chunk weighting

The abstract chunk encodes the paper's core contribution in ~150 words. During retrieval, abstract chunks naturally score high for topic-level queries. If you want to *boost* them explicitly, you can store a `boost_factor` in the Qdrant payload and apply it in post-processing — that's Exercise 3.

In [ ]:
class EmbeddingEngine:
    '''
    Wraps BGE-small with batch encoding and the BGE query instruction prefix.

    Documents are encoded without prefix.
    Queries are encoded with the instruction prefix for better retrieval.
    All embeddings are L2-normalised → cosine similarity = dot product.
    '''

    QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

    def __init__(self, model_name: str = "BAAI/bge-small-en-v1.5"):
        print(f"⏳ Loading: {model_name}")
        self.model      = SentenceTransformer(model_name)
        self.model_name = model_name
        self.dim        = self.model.get_embedding_dimension()
        print(f"✅ Loaded | dim={self.dim}")

    def embed_documents(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        '''Encode document chunks — no prefix.'''
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )

    def embed_query(self, query: str) -> np.ndarray:
        '''Encode a search query — with BGE instruction prefix.'''
        return self.model.encode(
            self.QUERY_PREFIX + query,
            normalize_embeddings=True,
        )


# ── Load model and embed all chunks ─────────────────────────────────────────
embedder   = EmbeddingEngine()
texts      = [c["text"] for c in chunks]
embeddings = embedder.embed_documents(texts)

# Attach embeddings to chunks
for chunk, emb in zip(chunks, embeddings):
    chunk["embedding"] = emb.tolist()

print(f"\n📐 Embedding matrix : {embeddings.shape}")
print(f"   Vectors          : {len(chunks)} × {embedder.dim}D float32")

# ── Sanity check ─────────────────────────────────────────────────────────────
test_query = "multi-head attention mechanism"
qvec       = embedder.embed_query(test_query)
scores     = embeddings @ qvec
top5       = np.argsort(scores)[::-1][:5]

print(f"\n🔎 Top-5 for '{test_query}':")
for rank, i in enumerate(top5, 1):
    ctype = chunks[i]["chunk_type"]
    print(f"  [{rank}] score={scores[i]:.4f} | {ctype} | pg {chunks[i]['page_num']} | {chunks[i]['text'][:80]}...")

## Section 7 — Qdrant Vector Store

Qdrant stores the embedding vectors and exposes approximate nearest-neighbour search. We run it in **in-memory mode** — no Docker, no installation beyond `pip`.

### What Qdrant stores

Each **point** in Qdrant has three parts:
- `id` — an integer (the chunk's position in the list)
- `vector` — the 384D float32 embedding
- `payload` — a JSON dict of all chunk metadata (doc_id, page_num, section, chunk_type, text, …)

We store the full text and metadata in the payload so search results are self-contained — no second lookup needed for basic display.

### Payload-based filtering

Qdrant supports filtering search results by payload fields **before** the ANN search runs (not as a post-filter). This is critical for:
- `doc_id` filter — search within a specific paper only
- `chunk_type` filter — search only abstract chunks, or only body chunks
- Combined filters for advanced queries

### Production note

To connect to a real Qdrant server (e.g. via Docker Compose in production):
```python
client = QdrantClient(host="qdrant", port=6333)
```
Everything else stays the same.

In [ ]:
class VectorStore:
    '''
    Qdrant-backed vector store.
    Runs in-memory by default; swap to QdrantClient(host=...) for production.
    '''

    COLLECTION = "scientific_papers"

    def __init__(self, dim: int):
        self.client = QdrantClient(":memory:")
        self.dim    = dim
        self._create_collection()

    def _create_collection(self):
        self.client.recreate_collection(
            collection_name=self.COLLECTION,
            vectors_config=VectorParams(size=self.dim, distance=Distance.COSINE),
        )
        print(f"✅ Qdrant collection '{self.COLLECTION}' ready (dim={self.dim})")

    def upsert(self, chunks: List[Dict]) -> int:
        '''Insert/update chunks. Payload stores all fields except 'embedding'.'''
        points = []
        for i, chunk in enumerate(chunks):
            payload = {k: v for k, v in chunk.items() if k != "embedding"}
            points.append(PointStruct(id=i, vector=chunk["embedding"], payload=payload))

        for start in range(0, len(points), 100):
            self.client.upsert(
                collection_name=self.COLLECTION,
                points=points[start:start + 100],
            )
        return len(points)

    def search(
        self,
        query_vector: List[float],
        top_k: int = 5,
        doc_id: Optional[str] = None,
        chunk_type: Optional[str] = None,
    ) -> List[Dict]:
        '''
        Dense vector search with optional payload filters.

        doc_id     — restrict to a single document
        chunk_type — 'abstract' | 'body' | None (both)
        '''
        must_conditions = []
        if doc_id:
            must_conditions.append(
                FieldCondition(key="doc_id", match=MatchValue(value=doc_id))
            )
        if chunk_type:
            must_conditions.append(
                FieldCondition(key="chunk_type", match=MatchValue(value=chunk_type))
            )

        query_filter = Filter(must=must_conditions) if must_conditions else None

        results = self.client.query_points(
            collection_name=self.COLLECTION,
            query=query_vector,
            limit=top_k,
            query_filter=query_filter,
            with_payload=True,
        )

        # print(results)
        return [
            {"chunk_id": r.payload["chunk_id"], "score": round(r.score, 4),
             "payload": r.payload}
            for r in results.points
        ]


# ── Upsert ───────────────────────────────────────────────────────────────────
vector_store = VectorStore(dim=embedder.dim)
n = vector_store.upsert(chunks)
print(f"📦 Upserted {n} vectors")
# print(dir(vector_store.client))

# ── Test: filter to abstract chunks only ─────────────────────────────────────
qvec = embedder.embed_query("transformer sequence to sequence").tolist()
abstract_results = vector_store.search(qvec, top_k=3, chunk_type="abstract")
print(f"\n🔎 Abstract-only search:")
for r in abstract_results:
    print(f"  score={r['score']} | {r['payload']['text'][:100]}...")

## Section 8 — Neo4j — Paper Nodes, Author Nodes & Relationships

This is where the architectural change lives. In Week 5, the citation graph was embedded inside MongoDB documents as `cites[]` and `cited_by[]` arrays, and authors were just a flat list of strings on each document. Now both become **first-class graph data** in Neo4j.

### Role 1: Paper Registry
One `(:Paper)` node per ingested paper (or stub). Stores all bibliographic metadata as node properties.

### Role 2: Author Registry & Authorship Graph
One `(:Author)` node per unique author (de-duplicated by normalised name). Linked to papers via `[:WROTE]` relationships with an `order` property preserving authorship position (first author vs. middle vs. last).

```cypher
(:Author {name, name_norm})-[:WROTE {order}]->(:Paper)
```

### Role 3: Citation Graph
Citations are `[:CITES]` relationships with edge properties carrying the confidence score and match tier:

```cypher
(:Paper)-[:CITES {confidence, match_tier, ref_title}]->(:Paper)
```

### Role 4: Derived Co-authorship Graph
Co-authorship is a **derived** relationship — we don't write it during ingestion, we compute it after the fact from the `[:WROTE]` graph. This is a core graph-DB pattern: don't denormalise what you can project.

```cypher
(:Author)-[:CO_AUTHORED_WITH {paper_count}]-(:Author)
```

Undirected (notice the `-` instead of `->` in queries), with a count property tracking how many papers two authors share.

### Role 5: Chunk Index
*Unchanged from Week 5.* Chunks remain in MongoDB (`mongomock`) because they're text-heavy documents without meaningful graph structure between them. Forcing them into Neo4j would just be using it as a document store, which isn't what it's for.

### Schema

```
NODES
─────
(:Paper {
    doc_id:       string  (unique)
    status:       string  -- 'ingested' | 'stub'
    title:        string
    authors:      list<string>   -- kept for the resolver's Tier-4 author/year match
    year:         int
    venue:        string
    doi:          string
    abstract:     string
    keywords:     list<string>
    source:       string
    ingested_at:  string
})

(:Author {
    name:       string             -- canonical display form (e.g. "Ashish Vaswani")
    name_norm:  string  (unique)   -- normalised key for de-duplication
})

RELATIONSHIPS
─────────────
(:Author)-[:WROTE {
    order:     int   -- 1 = first author, 2 = second, ...
}]->(:Paper)

(:Paper)-[:CITES {
    confidence:  float    -- 0.0 to 1.0
    match_tier:  string   -- 'doi' | 'cite_key' | 'title_fuzzy' | ...
    ref_title:   string   -- first 120 chars (debug)
}]->(:Paper)

(:Author)-[:CO_AUTHORED_WITH {
    paper_count: int   -- number of papers shared
}]-(:Author)            -- undirected; we write a single directed edge with
                        --  source.name_norm < target.name_norm to avoid duplicates
```

### Why we *also* keep `authors` as a property on `(:Paper)`

The `CitationResolver`'s Tier-4 fallback matches on first-author surname + year. That logic lives inside `_author_year_match()` which iterates over papers of a given year and checks their author list. We keep `authors` as a denormalised list property on `(:Paper)` so that Tier 4 stays a single Cypher query rather than a 2-hop traversal. The `(:Author)` graph is the *canonical* model; the list property is a small index-friendly mirror.

In a production system you might rewrite Tier 4 as a graph query (`MATCH (p:Paper {year})-[:WROTE]-(:Author {name_norm})`), but for the lab this would be over-engineering.

### Constraints & Indexes

```cypher
CREATE CONSTRAINT paper_doc_id_unique     FOR (p:Paper)  REQUIRE p.doc_id    IS UNIQUE;
CREATE CONSTRAINT author_name_norm_unique FOR (a:Author) REQUIRE a.name_norm IS UNIQUE;
CREATE INDEX paper_doi    FOR (p:Paper)  ON (p.doi);
CREATE INDEX paper_year   FOR (p:Paper)  ON (p.year);
CREATE INDEX paper_status FOR (p:Paper)  ON (p.status);
```

The uniqueness constraint on `name_norm` doubles as our de-duplication mechanism — `MERGE (a:Author {name_norm: $norm})` will find the existing node if one exists, or create a new one otherwise.

### Sample queries (we'll use these in the bonus section)

```cypher
-- Outgoing citations with confidence ≥ 0.8 (unchanged from before)
MATCH (p:Paper {doc_id: $doc_id})-[c:CITES]->(cited:Paper)
WHERE c.confidence >= 0.8
RETURN cited.doc_id, c.confidence ORDER BY c.confidence DESC

-- Author productivity in the corpus  (NEW)
MATCH (a:Author)-[:WROTE]->(p:Paper {status: 'ingested'})
RETURN a.name, count(p) AS papers ORDER BY papers DESC

-- "Which authors do I cite most?" — 2-hop across WROTE + CITES  (NEW)
MATCH (me:Author {name_norm: $me})-[:WROTE]->(:Paper)
      -[:CITES]->(:Paper)<-[:WROTE]-(other:Author)
WHERE other <> me
RETURN other.name, count(*) AS cited_count ORDER BY cited_count DESC

-- Co-authorship distance (Erdős-style)  (NEW)
MATCH path = shortestPath(
  (a:Author {name_norm: $from})-[:CO_AUTHORED_WITH*..6]-(b:Author {name_norm: $to})
)
RETURN [n IN nodes(path) | n.name] AS chain, length(path) AS distance
```

These last three queries are essentially impossible without the graph model. That's the value of normalising authors out into their own nodes.

In [ ]:
class Neo4jMetadataStore:
    '''
    Neo4j-backed store for paper nodes, author nodes, and citation relationships.
    MongoDB (mongomock) is still used for the chunks collection, since
    chunks aren't graph entities.

    Neo4j model:
        (:Paper)  nodes               — one per paper (ingested or stub)
        (:Author) nodes               — one per unique author (de-duped by name_norm)
        (:Author)-[:WROTE {order}]->(:Paper)
        (:Paper) -[:CITES]->(:Paper)
        (:Author)-[:CO_AUTHORED_WITH {paper_count}]-(:Author)    -- derived

    MongoDB collections (unchanged from Week 5):
        chunks  — one record per text chunk (no embedding vectors)
    '''

    def __init__(
        self,
        neo4j_uri:      str,
        neo4j_user:     str,
        neo4j_password: str,
        mongo_uri:      Optional[str] = None,
        mongo_db_name:  str           = "rag_db",
    ):
        # ── Neo4j ─────────────────────────────────────────────────────────────
        self.driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
        self._init_schema()

        # ── MongoDB (chunks only) ─────────────────────────────────────────────
        if mongo_uri:
            from pymongo import MongoClient
            self.mongo_client = MongoClient(mongo_uri)
        else:
            self.mongo_client = mongomock.MongoClient()

        self.db     = self.mongo_client[mongo_db_name]
        self.chunks = self.db["chunks"]
        self.chunks.create_index("chunk_id", unique=True)
        self.chunks.create_index("doc_id")

        print(f"✅ Neo4jMetadataStore ready (Neo4j + mongomock for chunks)")

    def close(self):
        '''Close the Neo4j driver. Call before program exit.'''
        self.driver.close()

    # ── Schema setup ──────────────────────────────────────────────────────────

    def _init_schema(self):
        '''Create uniqueness constraints + indexes for fast resolver lookups.'''
        with self.driver.session() as s:
            # Papers
            s.run("CREATE CONSTRAINT paper_doc_id_unique IF NOT EXISTS "
                  "FOR (p:Paper) REQUIRE p.doc_id IS UNIQUE")
            s.run("CREATE INDEX paper_doi    IF NOT EXISTS FOR (p:Paper) ON (p.doi)")
            s.run("CREATE INDEX paper_year   IF NOT EXISTS FOR (p:Paper) ON (p.year)")
            s.run("CREATE INDEX paper_status IF NOT EXISTS FOR (p:Paper) ON (p.status)")
            # Authors
            s.run("CREATE CONSTRAINT author_name_norm_unique IF NOT EXISTS "
                  "FOR (a:Author) REQUIRE a.name_norm IS UNIQUE")

    def wipe(self):
        '''Delete ALL Paper + Author nodes and relationships. Useful for re-running the demo.'''
        with self.driver.session() as s:
            s.run("MATCH (n) WHERE n:Paper OR n:Author DETACH DELETE n")
        self.chunks.delete_many({})
        print("🧹 Wiped Neo4j (:Paper, :Author nodes + relationships) and chunks collection")

    # ── Name normalisation ────────────────────────────────────────────────────

    @staticmethod
    def _norm_name(name: str) -> str:
        '''
        Normalise an author name for de-duplication.

        Handles common BibTeX styles:
          - "Vaswani, Ashish"   → "ashish vaswani"  (swap surname-first form)
          - "Ashish Vaswani"    → "ashish vaswani"
          - "A. Vaswani"        → "a vaswani"
          - "Vaswani, A."       → "a vaswani"

        This is intentionally simple — production would use a fuzzy
        author-resolution step (Tier 4-ish for authors). The lab focuses
        on the graph model, not on author disambiguation.
        '''
        if not name or not name.strip():
            return ""
        n = name.strip()
        # "Surname, Given" → "Given Surname"
        if "," in n:
            parts = [p.strip() for p in n.split(",", 1)]
            if len(parts) == 2:
                n = f"{parts[1]} {parts[0]}"
        # Lowercase, strip punctuation, collapse whitespace
        n = n.lower()
        n = re.sub(r"[^a-z0-9\s]", " ", n)
        n = re.sub(r"\s+", " ", n).strip()
        return n

    # ── Author + WROTE upsert ─────────────────────────────────────────────────

    def _upsert_authors_and_wrote_edges(self, doc_id: str, authors: List[str]):
        '''
        For each author string in the paper's author list:
          1. Compute the normalised key
          2. MERGE the (:Author) node by name_norm  (de-duplicates "A. Vaswani" vs "Ashish Vaswani")
          3. MERGE the (:Author)-[:WROTE {order}]->(:Paper) edge

        Author order is preserved as a property on the relationship —
        critical because first-author position has meaning in academic conventions.

        Idempotent: re-ingesting the same paper updates the order property
        but doesn't create duplicate edges.
        '''
        if not authors:
            return

        # Build the rows in Python so we can pass a single UNWIND payload
        rows = []
        for i, raw_name in enumerate(authors, start=1):
            norm = self._norm_name(raw_name)
            if not norm:
                continue
            rows.append({"name": raw_name.strip(), "name_norm": norm, "order": i})

        if not rows:
            return

        with self.driver.session() as s:
            s.run(
                """
                MATCH (p:Paper {doc_id: $doc_id})
                UNWIND $rows AS row
                MERGE (a:Author {name_norm: row.name_norm})
                  ON CREATE SET a.name = row.name
                MERGE (a)-[w:WROTE]->(p)
                  SET w.order = row.order
                """,
                doc_id = doc_id,
                rows   = rows,
            )

    # ── Document operations ───────────────────────────────────────────────────

    def upsert_document(self, doc: DocMetadata) -> str:
        '''
        Insert or update a (:Paper) node. If the node previously had
        status='stub', incoming [:CITES] relationships are preserved
        automatically (they live on the graph, not on the node).

        After the node is upserted, we also create/refresh the (:Author)
        nodes and [:WROTE] edges based on the paper's author list.
        '''
        d = doc.to_dict()
        d["status"] = "ingested"   # promoting from stub if applicable

        with self.driver.session() as s:
            s.run(
                """
                MERGE (p:Paper {doc_id: $doc_id})
                SET p += $props
                """,
                doc_id = doc.doc_id,
                props  = d,
            )

        # Build the author graph for this paper
        self._upsert_authors_and_wrote_edges(doc.doc_id, doc.authors)
        return doc.doc_id

    def upsert_stub(
        self,
        doc_id:  str,
        title:   str,
        authors: List[str],
        year:    Optional[int],
        doi:     str,
        venue:   str,
    ):
        '''
        Create or update a stub (:Paper) node. Stubs are real nodes
        with status='stub' — they exist so we can hang [:CITES] edges
        on them while waiting for the real PDF+.bib to be ingested.

        If a node with this doc_id already exists, we leave its
        properties alone (MERGE ON CREATE) — we don't want a later stub
        encounter to overwrite a fully-ingested record.

        We also create (:Author) nodes + [:WROTE] edges for any authors
        the reference parser managed to extract. Authors from references
        are often partial ("A. Vaswani") — that's OK; if the paper is
        later ingested with full author names, the upsert will merge
        them by name_norm.
        '''
        with self.driver.session() as s:
            s.run(
                """
                MERGE (p:Paper {doc_id: $doc_id})
                ON CREATE SET
                    p.status      = 'stub',
                    p.title       = $title,
                    p.authors     = $authors,
                    p.year        = $year,
                    p.doi         = $doi,
                    p.venue       = $venue,
                    p.abstract    = '',
                    p.keywords    = [],
                    p.source      = '',
                    p.ingested_at = $ingested_at
                """,
                doc_id      = doc_id,
                title       = title,
                authors     = authors,
                year        = year,
                doi         = doi,
                venue       = venue,
                ingested_at = datetime.utcnow().isoformat(),
            )

        # Hang authors off the stub too — even partial author info is useful for the graph
        self._upsert_authors_and_wrote_edges(doc_id, authors)

    def upgrade_to_ingested(self, doc_id: str, full_meta: DocMetadata):
        '''
        Upgrade a stub node to a full ingested record.
        Incoming [:CITES] relationships are preserved automatically.
        Author nodes are upserted again — any new full names will merge
        with the partial names already created when this was a stub.
        '''
        existing = self.get_document(doc_id)
        if not existing:
            return

        d = full_meta.to_dict()
        d["status"] = "ingested"

        with self.driver.session() as s:
            s.run(
                "MATCH (p:Paper {doc_id: $doc_id}) SET p += $props",
                doc_id = doc_id,
                props  = d,
            )
        self._upsert_authors_and_wrote_edges(doc_id, full_meta.authors)

        incoming = self.count_cited_by(doc_id)
        print(f"  ⬆️  Upgraded stub '{doc_id}' → ingested "
              f"(preserved {incoming} incoming :CITES edges)")

    def get_document(self, doc_id: str) -> Optional[Dict]:
        '''Fetch a single paper by doc_id.'''
        with self.driver.session() as s:
            rec = s.run(
                "MATCH (p:Paper {doc_id: $doc_id}) RETURN p",
                doc_id=doc_id,
            ).single()
            return dict(rec["p"]) if rec else None

    def find_by_doi(self, doi: str) -> Optional[Dict]:
        '''Resolver Tier 1 lookup.'''
        if not doi:
            return None
        with self.driver.session() as s:
            rec = s.run(
                "MATCH (p:Paper {doi: $doi}) RETURN p LIMIT 1",
                doi=doi,
            ).single()
            return dict(rec["p"]) if rec else None

    def find_by_year(self, year: int) -> List[Dict]:
        '''Resolver Tier 4 helper — all papers in a given year.'''
        with self.driver.session() as s:
            recs = s.run(
                "MATCH (p:Paper {year: $year}) RETURN p",
                year=year,
            )
            return [dict(r["p"]) for r in recs]

    def iter_titles(self) -> List[Dict]:
        '''
        Resolver Tier 3 helper — yields {doc_id, title} for every paper.
        At lab scale this is fine; production would use a full-text index.
        '''
        with self.driver.session() as s:
            recs = s.run(
                "MATCH (p:Paper) WHERE p.title IS NOT NULL "
                "RETURN p.doc_id AS doc_id, p.title AS title"
            )
            return [{"doc_id": r["doc_id"], "title": r["title"]} for r in recs]

    def list_documents(self, status: Optional[str] = None) -> List[Dict]:
        '''List papers, optionally filtered by status ('ingested' | 'stub').'''
        with self.driver.session() as s:
            if status:
                recs = s.run(
                    "MATCH (p:Paper {status: $status}) RETURN p",
                    status=status,
                )
            else:
                recs = s.run("MATCH (p:Paper) RETURN p")
            return [dict(r["p"]) for r in recs]

    # ── Citation graph operations ─────────────────────────────────────────────

    def create_cites_edge(
        self,
        citing_doc_id: str,
        cited_doc_id:  str,
        confidence:    float,
        match_tier:    str,
        ref_title:     str = "",
    ):
        '''
        Create (or update) a [:CITES] relationship.
        Idempotent: re-ingesting the same paper doesn't create duplicate edges.

        Note: MERGE on the relationship pattern means
        "find or create exactly one [:CITES] between these two nodes".
        '''
        with self.driver.session() as s:
            s.run(
                """
                MATCH (citing:Paper {doc_id: $citing})
                MATCH (cited:Paper  {doc_id: $cited})
                MERGE (citing)-[c:CITES]->(cited)
                SET c.confidence = $confidence,
                    c.match_tier = $match_tier,
                    c.ref_title  = $ref_title
                """,
                citing     = citing_doc_id,
                cited      = cited_doc_id,
                confidence = confidence,
                match_tier = match_tier,
                ref_title  = ref_title,
            )

    def get_citations(
        self, doc_id: str, min_confidence: float = 0.0
    ) -> List[Dict]:
        '''
        Return outgoing [:CITES] edges from this paper, optionally
        filtered by confidence. Each result includes the cited
        paper's doc_id plus the edge properties.
        '''
        with self.driver.session() as s:
            recs = s.run(
                """
                MATCH (p:Paper {doc_id: $doc_id})-[c:CITES]->(cited:Paper)
                WHERE c.confidence >= $min_conf
                RETURN cited.doc_id   AS doc_id,
                       c.confidence   AS confidence,
                       c.match_tier   AS match_tier,
                       c.ref_title    AS ref_title
                ORDER BY c.confidence DESC
                """,
                doc_id   = doc_id,
                min_conf = min_confidence,
            )
            return [dict(r) for r in recs]

    def get_cited_by(self, doc_id: str) -> List[str]:
        '''Return list of doc_ids that cite this paper.'''
        with self.driver.session() as s:
            recs = s.run(
                """
                MATCH (citing:Paper)-[:CITES]->(p:Paper {doc_id: $doc_id})
                RETURN citing.doc_id AS doc_id
                """,
                doc_id=doc_id,
            )
            return [r["doc_id"] for r in recs]

    def count_cited_by(self, doc_id: str) -> int:
        '''Fast count of incoming :CITES edges.'''
        with self.driver.session() as s:
            rec = s.run(
                """
                MATCH (citing:Paper)-[:CITES]->(p:Paper {doc_id: $doc_id})
                RETURN count(citing) AS n
                """,
                doc_id=doc_id,
            ).single()
            return rec["n"] if rec else 0

    # ── Co-authorship: derived from WROTE edges ───────────────────────────────

    def refresh_coauthorship(self) -> int:
        '''
        Rebuild the [:CO_AUTHORED_WITH] graph from current [:WROTE] edges.

        Strategy:
          1. Delete all existing :CO_AUTHORED_WITH edges (cheap idempotent rebuild).
          2. For every pair (a, b) of authors sharing at least one paper,
             write a single directed edge from the lexicographically smaller
             name_norm to the larger one, with paper_count = how many papers
             they share.

        Why a single directed edge for an undirected relationship?
        Neo4j stores every relationship with a direction internally, but
        queries can match without specifying one: `MATCH (a)-[:CO_AUTHORED_WITH]-(b)`.
        Writing only one direction avoids storing the same fact twice.

        Returns the number of co-authorship edges created.
        '''
        with self.driver.session() as s:
            # Wipe existing co-authorship edges
            s.run("MATCH ()-[c:CO_AUTHORED_WITH]-() DELETE c")

            # Recompute
            rec = s.run(
                """
                MATCH (a:Author)-[:WROTE]->(p:Paper)<-[:WROTE]-(b:Author)
                WHERE a.name_norm < b.name_norm
                WITH a, b, count(DISTINCT p) AS paper_count
                MERGE (a)-[c:CO_AUTHORED_WITH]->(b)
                SET c.paper_count = paper_count
                RETURN count(c) AS n
                """
            ).single()
            return rec["n"] if rec else 0

    # ── Chunk operations (mongomock — unchanged from Week 5) ──────────────────

    def upsert_chunks(self, chunks: List[Dict]) -> int:
        '''Store chunk metadata (without embedding vectors).'''
        for chunk in chunks:
            doc = {k: v for k, v in chunk.items() if k != "embedding"}
            self.chunks.update_one(
                {"chunk_id": doc["chunk_id"]},
                {"$set": doc},
                upsert=True,
            )
        return len(chunks)

    def get_chunks_by_doc(self, doc_id: str) -> List[Dict]:
        results = list(self.chunks.find({"doc_id": doc_id}).sort("chunk_seq", 1))
        return [{k: v for k, v in r.items() if k != "_id"} for r in results]

    # ── Stats ─────────────────────────────────────────────────────────────────

    def stats(self) -> Dict:
        with self.driver.session() as s:
            rec = s.run(
                """
                MATCH (p:Paper)
                RETURN
                  sum(CASE WHEN p.status = 'ingested' THEN 1 ELSE 0 END) AS ingested,
                  sum(CASE WHEN p.status = 'stub'     THEN 1 ELSE 0 END) AS stubs,
                  count(p) AS total
                """
            ).single()
            cites_edge = s.run("MATCH ()-[c:CITES]->() RETURN count(c) AS n").single()
            author_count = s.run("MATCH (a:Author) RETURN count(a) AS n").single()
            wrote_edge   = s.run("MATCH ()-[w:WROTE]->() RETURN count(w) AS n").single()
            coauth_edge  = s.run("MATCH ()-[c:CO_AUTHORED_WITH]->() RETURN count(c) AS n").single()
        return {
            "ingested_papers":      rec["ingested"] or 0,
            "stub_papers":          rec["stubs"] or 0,
            "total_papers":         rec["total"] or 0,
            "cites_edges":          cites_edge["n"] or 0,
            "authors":              author_count["n"] or 0,
            "wrote_edges":          wrote_edge["n"] or 0,
            "co_authored_edges":    coauth_edge["n"] or 0,
            "total_chunks":         self.chunks.count_documents({}),
        }


# ── Initialise store ──────────────────────────────────────────────────────────
metadata_store = Neo4jMetadataStore(
    neo4j_uri      = NEO4J_URI,
    neo4j_user     = NEO4J_USER,
    neo4j_password = NEO4J_PASSWORD,
)

# Optional: start clean. Comment out if you want to keep accumulated state
# across notebook re-runs.
metadata_store.wipe()

print(metadata_store.stats())

## Section 9 — Full Ingestion Pipeline

Now we assemble all the components into a single `IngestionPipeline` class. With the author graph added, there's one more step at the end: refreshing the derived `[:CO_AUTHORED_WITH]` edges.

### The Ingestion Sequence

```
ingest(pdf_path, bib_path)
│
├── 1. BibParser.parse(bib_path)
│       → DocMetadata (title, authors, year, venue, doi, abstract, …)
│
├── 2. Check Neo4j: does this doc_id already exist as a stub?
│       → If yes, the upcoming upsert_document() will flip status to 'ingested'
│         and all existing incoming [:CITES] edges are preserved automatically.
│
├── 3. PDFTextExtractor.extract_pages(pdf_path)
│       → List of page dicts with text and headings
│
├── 4. SlidingWindowChunker.chunk_pages(pages, abstract)
│       → List of chunk dicts (includes abstract chunk)
│
├── 5. EmbeddingEngine.embed_documents(texts)
│       → numpy array of 384D vectors, attached to chunks
│
├── 6. VectorStore.upsert(chunks)
│       → chunks indexed in Qdrant
│
├── 7. Neo4jMetadataStore.upsert_document(meta)
│       → writes (:Paper) node, (:Author) nodes, and [:WROTE] edges
│   Neo4jMetadataStore.upsert_chunks(chunks)
│       → writes chunks to MongoDB
│
├── 8. ReferenceExtractor.extract(pdf_path)
│       → List[RawReference] (via embedded .bib/.bbl or regex)
│
├── 9. CitationResolver.resolve_all(raw_refs, doc_id)
│       → For each reference:
│           Match against Neo4j (tiers 1-4) or create stub (:Paper) node
│           Create [:CITES] edge with confidence + match_tier
│           Stub creation ALSO creates (:Author) nodes from partial author info
│       → List of citation edge dicts (just for the summary)
│
└── 10. Neo4jMetadataStore.refresh_coauthorship()
        → Recompute [:CO_AUTHORED_WITH] edges from the current [:WROTE] graph
        → Cheap: a single Cypher statement, rebuild rather than incremental
```

### Why refresh coauthorship as a separate step?

Co-authorship is a **derived** relationship — it's a function of `[:WROTE]` edges. We have two options:

1. **Incremental**: every time we ingest a paper, write new `[:CO_AUTHORED_WITH]` edges between its authors. Fast per-ingest but complicated when authors merge later (e.g., "A. Vaswani" merges with "Ashish Vaswani").
2. **Rebuild on demand**: wipe all `[:CO_AUTHORED_WITH]` edges and recompute from the current `[:WROTE]` graph in one Cypher statement.

We use **option 2** because it's simpler, idempotent, and at lab scale the rebuild cost is negligible. Production systems with millions of authors would use option 1 with careful merge-handling logic — but that's an over-engineering rabbit hole for this lab.

### What disappeared from Week 5

Two steps that were necessary with MongoDB are gone now:

- **No more `update_cites` step**: in Week 5 we had to write the `cites[]` array back onto the document. Now `[:CITES]` edges are created directly by the resolver — they're not array fields, they're graph relationships.
- **No more manual `cited_by` write-back inside the resolver**: in Week 5 we had to maintain both sides of every citation manually (`$addToSet` on `cited_by`). Now there's only one side — the edge — and Neo4j stores its traversal both ways automatically.

### Idempotency

The pipeline is safe to run multiple times on the same paper:
- Neo4j uses `MERGE` for both nodes (on `doc_id`) and `[:CITES]` edges (on the pattern between two papers) — no duplicates.
- Qdrant `recreate_collection` in `VectorStore` resets on restart (production would use `upsert` by point ID).
- Chunks use MongoDB `upsert` keyed on `chunk_id`.

In [ ]:
class IngestionPipeline:
    '''
    Orchestrates the full ingestion sequence for a PDF + .bib pair.

    Components:
        BibParser            → structured metadata from .bib
        PDFTextExtractor     → body text from PDF
        SlidingWindowChunker → overlapping text chunks
        EmbeddingEngine      → BGE-small 384D vectors
        VectorStore          → Qdrant upsert
        Neo4jMetadataStore   → Neo4j (:Paper) nodes + chunk records in MongoDB
        ReferenceExtractor   → reference list (embedded or regex)
        CitationResolver     → match references → [:CITES] edges + stubs
    '''

    def __init__(
        self,
        embedder:       EmbeddingEngine,
        vector_store:   VectorStore,
        metadata_store: Neo4jMetadataStore,
        layout_mode:    str = "text",
        chunk_size:     int = 400,
        chunk_overlap:  int = 50,
    ):
        self.embedder        = embedder
        self.vector_store    = vector_store
        self.metadata_store  = metadata_store

        self.bib_parser      = BibParser()
        self.text_extractor  = PDFTextExtractor(layout_mode=layout_mode)
        self.chunker         = SlidingWindowChunker(chunk_size, chunk_overlap)
        self.ref_extractor   = ReferenceExtractor()
        self.resolver        = CitationResolver(metadata_store)

    def ingest(self, pdf_path: str, bib_path: str) -> Dict:
        '''
        Ingest a PDF + .bib pair. Returns a summary dict.
        Raises if either file is missing.
        '''
        pdf_path = str(pdf_path)
        bib_path = str(bib_path)

        assert Path(pdf_path).exists(), f"PDF not found: {pdf_path}"
        assert Path(bib_path).exists(), f".bib not found: {bib_path}"

        print(f"\n{'═'*60}")
        print(f"📥 Ingesting: {Path(pdf_path).stem}")
        print(f"{'═'*60}")

        # ── Step 1: Parse .bib ────────────────────────────────────────────────
        print("  [1/9] Parsing .bib …")
        meta = self.bib_parser.parse(bib_path)
        print(f"        doc_id={meta.doc_id} | {meta.title[:60]}")

        # ── Step 2: Check for existing stub ───────────────────────────────────
        print("  [2/9] Checking for existing stub …")
        existing = self.metadata_store.get_document(meta.doc_id)
        if existing and existing.get("status") == "stub":
            inc = self.metadata_store.count_cited_by(meta.doc_id)
            print(f"        ↑ Stub '{meta.doc_id}' will be promoted "
                  f"(preserves {inc} incoming :CITES edges)")

        # ── Step 3: Extract text pages ────────────────────────────────────────
        print("  [3/9] Extracting PDF text …")
        pages = self.text_extractor.extract_pages(pdf_path, meta.doc_id)
        print(f"        {len(pages)} usable pages")

        # ── Step 4: Chunk ─────────────────────────────────────────────────────
        print("  [4/9] Chunking …")
        chunks = self.chunker.chunk_pages(pages, abstract=meta.abstract)
        print(f"        {len(chunks)} chunks ({sum(1 for c in chunks if c['chunk_type']=='abstract')} abstract)")

        # ── Step 5: Embed ─────────────────────────────────────────────────────
        print("  [5/9] Embedding …")
        texts      = [c["text"] for c in chunks]
        embeddings = self.embedder.embed_documents(texts)
        for c, e in zip(chunks, embeddings):
            c["embedding"] = e.tolist()

        # ── Step 6: Upsert vectors ────────────────────────────────────────────
        print("  [6/9] Upserting vectors into Qdrant …")
        self.vector_store.upsert(chunks)

        # ── Step 7: Write metadata + chunks ───────────────────────────────────
        print("  [7/9] Writing (:Paper) node, (:Author) nodes, [:WROTE] edges + chunks …")
        self.metadata_store.upsert_document(meta)
        self.metadata_store.upsert_chunks(chunks)

        # ── Step 8: Extract + resolve references ──────────────────────────────
        print("  [8/9] Extracting + resolving references …")
        raw_refs, ref_source = self.ref_extractor.extract(pdf_path)
        print(f"        {len(raw_refs)} references found via '{ref_source}'")

        citation_edges = self.resolver.resolve_all(raw_refs, meta.doc_id)

        resolved = sum(1 for e in citation_edges if e["match_tier"] != "stub_created")
        stubs    = sum(1 for e in citation_edges if e["match_tier"] == "stub_created")
        print(f"        {resolved} resolved | {stubs} new stub nodes")

        # Note: no separate update_cites call needed — edges were written
        # directly by the resolver as it ran.

        # ── Step 9: Refresh derived co-authorship graph ───────────────────────
        print("  [9/9] Refreshing co-authorship graph …")
        n_coauth = self.metadata_store.refresh_coauthorship()
        print(f"        {n_coauth} :CO_AUTHORED_WITH edges in graph")

        summary = {
            "doc_id":          meta.doc_id,
            "title":           meta.title,
            "status":          "ingested",
            "pages":           len(pages),
            "chunks":          len(chunks),
            "references":      len(raw_refs),
            "citations_resolved": resolved,
            "stubs_created":   stubs,
            "ref_source":      ref_source,
            "coauthor_edges":  n_coauth,
        }
        print(f"\n✅ Done: {summary}")
        return summary

    def ingest_directory(self, directory: str) -> List[Dict]:
        '''
        Ingest all PDF+.bib pairs in a directory.
        Files must share the same stem: paper.pdf + paper.bib
        Skips PDFs with no matching .bib (logs a warning).
        '''
        dir_path = Path(directory)
        summaries = []

        pdf_files = sorted(dir_path.glob("*.pdf"))
        print(f"📂 Found {len(pdf_files)} PDF files in '{directory}'")

        for pdf in pdf_files:
            bib = pdf.with_suffix(".bib")
            if not bib.exists():
                print(f"  ⚠️  No .bib for {pdf.name} — skipping")
                continue
            try:
                summary = self.ingest(str(pdf), str(bib))
                summaries.append(summary)
            except Exception as e:
                print(f"  ❌ Error ingesting {pdf.name}: {e}")

        return summaries


# ── Build the pipeline ────────────────────────────────────────────────────────
pipeline = IngestionPipeline(
    embedder       = embedder,
    vector_store   = vector_store,
    metadata_store = metadata_store,
)

# ── Ingest the demo paper ─────────────────────────────────────────────────────
summary = pipeline.ingest(PDF_PATH, "./papers/VaswaniSPUJGKP17.bib")

print(f"\n📊 Neo4j + chunk stats after ingestion:")
print(json.dumps(metadata_store.stats(), indent=2))

In [ ]:
summary2 = pipeline.ingest("./papers/devlin2018bert.pdf", "./papers/devlin2018bert.bib")

print(f"\n📊 Neo4j + chunk stats after ingestion:")
print(json.dumps(metadata_store.stats(), indent=2))

In [ ]:
# ── Inspect the citation graph — directly in Cypher ──────────────────────────
from IPython.display import Markdown, display

print(meta.doc_id)

# Helper: run a Cypher query and return rows as dicts
def cypher(query: str, **params):
    with metadata_store.driver.session() as s:
        return [dict(r) for r in s.run(query, **params)]

# ── 1. The paper node itself ─────────────────────────────────────────────────
paper = cypher("""
    MATCH (p:Paper {doc_id: $doc_id})
    RETURN p.doc_id  AS doc_id,
           p.title   AS title,
           p.status  AS status,
           p.authors AS authors,
           p.year    AS year
""", doc_id=meta.doc_id)[0]

print(f"📋 Paper node: {paper['title']}")
print(f"   Status   : {paper['status']}")
print(f"   Authors  : {', '.join(paper['authors'][:3])}")
print(f"   Year     : {paper['year']}")

# ── 2. Outgoing :CITES edges with their target paper ─────────────────────────
edges = cypher("""
    MATCH (p:Paper {doc_id: $doc_id})-[c:CITES]->(cited:Paper)
    RETURN cited.doc_id   AS doc_id,
           cited.title    AS title,
           cited.status   AS status,
           c.confidence   AS confidence,
           c.match_tier   AS match_tier
    ORDER BY c.confidence DESC
""", doc_id=meta.doc_id)

print(f"\n📤 Outgoing :CITES edges ({len(edges)}):")
for e in edges[:8]:
    title = (e['title'] or "")[:50]
    print(f"  -[:CITES {{conf={e['confidence']:.2f}, tier={e['match_tier']}}}]-> "
          f"{e['doc_id']:<30} '{title}'")
if len(edges) > 8:
    print(f"  … and {len(edges)-8} more")

# ── 3. Incoming :CITES edges ─────────────────────────────────────────────────
incoming = cypher("""
    MATCH (citing:Paper)-[c:CITES]->(p:Paper {doc_id: $doc_id})
    RETURN citing.doc_id AS doc_id, c.confidence AS confidence
""", doc_id=meta.doc_id)

print(f"\n📥 Incoming :CITES edges ({len(incoming)}):")
for r in incoming:
    print(f"  <-[:CITES {{conf={r['confidence']:.2f}}}]- {r['doc_id']}")

# ── 4. Authors of this paper via :WROTE edges (with order) ───────────────────
authors = cypher("""
    MATCH (a:Author)-[w:WROTE]->(p:Paper {doc_id: $doc_id})
    RETURN a.name      AS name,
           a.name_norm AS name_norm,
           w.order     AS order
    ORDER BY w.order
""", doc_id=meta.doc_id)

print(f"\n🖋️  :WROTE edges into this paper ({len(authors)}):")
for a in authors:
    print(f"  ({a['order']}) {a['name']:<30} (name_norm='{a['name_norm']}')")

# ── 5. Stub :Paper nodes ─────────────────────────────────────────────────────
stubs = cypher("""
    MATCH (p:Paper {status: 'stub'})
    RETURN p.doc_id AS doc_id, p.title AS title
""")

print(f"\n🔲 Stub :Paper nodes in the graph: {len(stubs)}")
for s in stubs[:5]:
    print(f"  {s['doc_id']:<35} title='{(s['title'] or '')[:50]}'")
if len(stubs) > 5:
    print(f"  … and {len(stubs)-5} more")

## Section 10 — BM25 Sparse Retrieval

Dense retrieval (BGE-small) is excellent at capturing **semantic similarity** — it finds passages that *mean* the same thing even if they use different words. But it struggles with **exact keyword matching**. If someone searches for `"BLEU score"` or `"ReLU activation"`, a dense model may return passages about evaluation metrics or activation functions in general, missing the specific term.

**BM25** (Best Match 25) fills this gap. It's the probabilistic retrieval model underlying Elasticsearch and Apache Solr, and it excels at exact and near-exact keyword retrieval.

### BM25 formula

```
BM25(q, d) = Σ IDF(qi) × (f(qi, d) × (k1+1)) / (f(qi, d) + k1 × (1 - b + b×|d|/avgdl))
```

Where:
- `f(qi, d)` — term frequency of query term i in document d
- `IDF(qi)` — inverse document frequency (penalises common terms)
- `|d|` — document length, `avgdl` — average document length
- `k1=1.5`, `b=0.75` — tunable saturation parameters

### Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

Neither dense nor sparse search dominates across all query types. The hybrid approach combines both:

```
RRF(d) = Σ  1 / (k + rank_i(d))      where k = 60 (Cormack et al., 2009)
```

Each document's RRF score is the sum of `1/(60 + rank)` across both result lists. Documents that appear high in *both* lists get a strong boost. Documents that only appear in one list still contribute, but less.

**Why RRF over weighted sum?** RRF is score-agnostic — it only uses rank position, so there's no need to normalise the very different score scales of BM25 and cosine similarity.

In [ ]:
class BM25Retriever:
    '''
    BM25 sparse retriever over all indexed chunks.
    Rebuilt whenever new documents are ingested.
    '''

    def __init__(self, chunks: List[Dict]):
        self.chunks    = chunks
        self.chunk_ids = [c["chunk_id"] for c in chunks]

        tokenized  = [self._tokenize(c["text"]) for c in chunks]
        self.bm25  = BM25Okapi(tokenized)
        print(f"✅ BM25 index built over {len(chunks)} chunks")

    def _tokenize(self, text: str) -> List[str]:
        '''Lowercase + extract alphabetic tokens of length ≥ 2.'''
        return re.findall(r"\b[a-z]{2,}\b", text.lower())

    def search(
        self,
        query: str,
        top_k: int = 5,
        doc_id: Optional[str] = None,
    ) -> List[Dict]:
        '''
        BM25 search with optional doc_id filter (post-filter).
        Returns list of {chunk_id, score, payload}.
        '''
        tokens = self._tokenize(query)
        scores = self.bm25.get_scores(tokens)

        # Apply doc_id filter before sorting
        if doc_id:
            for i, chunk in enumerate(self.chunks):
                if chunk["doc_id"] != doc_id:
                    scores[i] = 0.0

        top_idx = np.argsort(scores)[::-1][:top_k]
        return [
            {
                "chunk_id": self.chunk_ids[i],
                "score":    round(float(scores[i]), 4),
                "payload":  {k: v for k, v in self.chunks[i].items() if k != "embedding"},
            }
            for i in top_idx if scores[i] > 0
        ]


def reciprocal_rank_fusion(
    dense_results:  List[Dict],
    sparse_results: List[Dict],
    k: int = 60,
    top_k: int = 5,
) -> List[Dict]:
    '''
    Combine dense and sparse results using Reciprocal Rank Fusion.

    RRF score for document d:
        RRF(d) = Σ  1 / (k + rank_i(d))

    The constant k=60 prevents high-ranked documents from dominating
    when the score gap between rank 1 and rank 2 is very large.
    '''
    rrf_scores: Dict[str, float] = {}
    payloads:   Dict[str, Dict]  = {}

    for rank, hit in enumerate(dense_results, start=1):
        cid = hit["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank)
        payloads[cid]   = hit["payload"]

    for rank, hit in enumerate(sparse_results, start=1):
        cid = hit["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank)
        payloads[cid]   = hit.get("payload", {})

    sorted_ids = sorted(rrf_scores, key=rrf_scores.__getitem__, reverse=True)

    return [
        {
            "chunk_id":  cid,
            "rrf_score": round(rrf_scores[cid], 6),
            "payload":   payloads[cid],
        }
        for cid in sorted_ids[:top_k]
    ]


# ── Build BM25 index ─────────────────────────────────────────────────────────
bm25_retriever = BM25Retriever(chunks)

# ── Side-by-side: Dense vs BM25 vs Hybrid ────────────────────────────────────
query = "positional encoding sinusoidal"

qvec   = embedder.embed_query(query).tolist()
dense  = vector_store.search(qvec, top_k=5)
sparse = bm25_retriever.search(query, top_k=5)
hybrid = reciprocal_rank_fusion(dense, sparse, top_k=5)

print(f"Query: '{query}'\n{'─'*70}")
print(f"{'DENSE':<33} | {'BM25':<33}")
print(f"{'─'*70}")
for d, s in zip(dense[:3], sparse[:3]):
    dt = d['payload']['text'][:30] + "..."
    st = s['payload']['text'][:30] + "..."
    print(f"{dt:<33} | {st:<33}")
print(f"\n🏆 Hybrid RRF Top-3:")
for i, r in enumerate(hybrid[:3], 1):
    print(f"  [{i}] rrf={r['rrf_score']:.5f} | {r['payload']['text'][:90]}...")

## Section 11 — FastAPI Search + Citation Endpoints

We expose the full pipeline as a REST API. The API has three groups of endpoints — **the surface is identical to Week 5** so any client code keeps working. The only thing that changed is what happens inside the citation endpoints (now Cypher queries against Neo4j instead of MongoDB array lookups).

### Search endpoints
| Method | Path | Description |
|---|---|---|
| `GET` | `/search` | Semantic + hybrid search across all papers |

Key parameters:
- `q` — the query string
- `mode` — `dense` \| `sparse` \| `hybrid` (default: `hybrid`)
- `top_k` — number of results (default: 5)
- `doc_id` — restrict search to a single document
- `chunk_type` — `abstract` \| `body` \| both (default: both)
- `min_confidence` — minimum citation confidence for citation traversal (0.0–1.0)

### Document endpoints
| Method | Path | Description |
|---|---|---|
| `GET` | `/documents` | List all documents (filter by status) |
| `GET` | `/document/{doc_id}` | Full record for one document |
| `GET` | `/document/{doc_id}/citations` | Outgoing citations with confidence scores |
| `GET` | `/document/{doc_id}/cited_by` | Incoming citations |

### Ingestion endpoints
| Method | Path | Description |
|---|---|---|
| `POST` | `/ingest` | Upload a PDF + `.bib` pair |
| `POST` | `/reconcile` | Re-run citation resolution for all stubs |

### The `min_confidence` parameter

On the `/document/{doc_id}/citations` endpoint:
```
GET /document/vaswani2017attention/citations?min_confidence=0.8
```

Now backed by Cypher: `MATCH (p)-[c:CITES]->(cited) WHERE c.confidence >= $min_conf ...`. Useful for:
- Downstream citation graph analysis (exclude low-quality edges)
- Building trust-ranked reading lists ("papers this paper definitely cites")

In [ ]:
from fastapi import FastAPI, HTTPException, UploadFile, File, Query
from fastapi.responses import JSONResponse
import uvicorn, tempfile, time

app = FastAPI(
    title        = "Scientific Paper RAG API",
    description  = "PDF+Bib ingestion → Neo4j citation graph → hybrid search",
    version      = "2.0.0-neo4j",
)

# ── Shared state ──────────────────────────────────────────────────────────────
app.state.pipeline      = pipeline
app.state.embedder      = embedder
app.state.vector_store  = vector_store
app.state.bm25          = bm25_retriever
app.state.metadata      = metadata_store
app.state.all_chunks    = chunks


# ─────────────────────────────────────────────────────────────────────────────
# Health
# ─────────────────────────────────────────────────────────────────────────────

@app.get("/", summary="Health check")
def root():
    stats = app.state.metadata.stats()
    return {"status": "ok", "service": "Scientific Paper RAG v2 (Neo4j)", **stats}


# ─────────────────────────────────────────────────────────────────────────────
# Search
# ─────────────────────────────────────────────────────────────────────────────

@app.get("/search", summary="Hybrid semantic search")
def search(
    q:              str            = Query(...,      description="Search query text"),
    top_k:          int            = Query(5,        description="Number of results"),
    mode:           str            = Query("hybrid", description="dense | sparse | hybrid"),
    doc_id:         Optional[str]  = Query(None,     description="Filter to one document"),
    chunk_type:     Optional[str]  = Query(None,     description="abstract | body"),
    min_confidence: float          = Query(0.0,      description="Min citation confidence (for future graph search)"),
):
    if not q.strip():
        raise HTTPException(400, "Query cannot be empty")
    if mode not in ("dense", "sparse", "hybrid"):
        raise HTTPException(400, f"mode must be dense|sparse|hybrid, got '{mode}'")

    dense_results = sparse_results = []

    if mode in ("dense", "hybrid"):
        qvec          = app.state.embedder.embed_query(q).tolist()
        dense_results = app.state.vector_store.search(
            qvec, top_k=top_k * 2, doc_id=doc_id, chunk_type=chunk_type
        )

    if mode in ("sparse", "hybrid"):
        sparse_results = app.state.bm25.search(q, top_k=top_k * 2, doc_id=doc_id)
        if chunk_type:
            sparse_results = [
                r for r in sparse_results
                if r["payload"].get("chunk_type") == chunk_type
            ]

    if mode == "dense":
        results = dense_results[:top_k]
    elif mode == "sparse":
        results = sparse_results[:top_k]
    else:
        results = reciprocal_rank_fusion(dense_results, sparse_results, top_k=top_k)

    return {
        "query":   q,
        "mode":    mode,
        "count":   len(results),
        "results": results,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Documents
# ─────────────────────────────────────────────────────────────────────────────

@app.get("/documents", summary="List all documents")
def list_documents(
    status: Optional[str] = Query(None, description="ingested | stub")
):
    docs = app.state.metadata.list_documents(status=status)
    return {"count": len(docs), "documents": docs}


@app.get("/document/{doc_id}", summary="Get one document")
def get_document(doc_id: str):
    doc = app.state.metadata.get_document(doc_id)
    if not doc:
        raise HTTPException(404, f"Document '{doc_id}' not found")
    return doc


@app.get("/document/{doc_id}/citations", summary="Outgoing citations")
def get_citations(
    doc_id:         str,
    min_confidence: float = Query(0.0, description="Minimum confidence threshold (0.0–1.0)"),
):
    '''
    Returns all papers this document cites — backed by a Cypher traversal
    of outgoing [:CITES] edges.
    Use min_confidence to filter out uncertain matches.
    Example: /document/vaswani2017attention/citations?min_confidence=0.8
    '''
    doc = app.state.metadata.get_document(doc_id)
    if not doc:
        raise HTTPException(404, f"Document '{doc_id}' not found")

    edges = app.state.metadata.get_citations(doc_id, min_confidence=min_confidence)

    # Enrich each edge with the cited paper's title and status
    enriched = []
    for edge in edges:
        cited = app.state.metadata.get_document(edge["doc_id"])
        enriched.append({
            **edge,
            "cited_title":  cited["title"] if cited else "Unknown",
            "cited_status": cited["status"] if cited else "unknown",
            "cited_year":   cited.get("year") if cited else None,
        })

    return {
        "doc_id":          doc_id,
        "min_confidence":  min_confidence,
        "citation_count":  len(enriched),
        "citations":       enriched,
    }


@app.get("/document/{doc_id}/cited_by", summary="Incoming citations")
def get_cited_by(doc_id: str):
    '''
    Returns all papers in the corpus that cite this document — backed by
    a Cypher traversal of incoming [:CITES] edges. For stub documents,
    this shows which ingested papers reference it.
    '''
    doc = app.state.metadata.get_document(doc_id)
    if not doc:
        raise HTTPException(404, f"Document '{doc_id}' not found")

    cited_by_ids = app.state.metadata.get_cited_by(doc_id)

    # Enrich with titles
    enriched = []
    for cid in cited_by_ids:
        citing = app.state.metadata.get_document(cid)
        enriched.append({
            "doc_id": cid,
            "title":  citing["title"] if citing else "Unknown",
            "year":   citing.get("year") if citing else None,
            "status": citing["status"] if citing else "unknown",
        })

    return {
        "doc_id":       doc_id,
        "cited_by_count": len(enriched),
        "cited_by":     enriched,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Ingestion
# ─────────────────────────────────────────────────────────────────────────────

@app.post("/ingest", summary="Ingest a PDF + .bib pair")
async def ingest(
    pdf:  UploadFile = File(..., description="The PDF file"),
    bib:  UploadFile = File(..., description="The matching .bib file"),
):
    '''
    Upload a PDF and its matching .bib file.
    Both files must have the same stem (e.g. paper.pdf + paper.bib).
    The pipeline runs ingestion and returns a summary.
    '''
    if not pdf.filename.endswith(".pdf"):
        raise HTTPException(400, "pdf must be a .pdf file")
    if not bib.filename.endswith(".bib"):
        raise HTTPException(400, "bib must be a .bib file")

    with tempfile.TemporaryDirectory() as tmp:
        pdf_path = Path(tmp) / pdf.filename
        bib_path = Path(tmp) / bib.filename

        pdf_path.write_bytes(await pdf.read())
        bib_path.write_bytes(await bib.read())

        try:
            summary = app.state.pipeline.ingest(str(pdf_path), str(bib_path))
        except Exception as e:
            raise HTTPException(500, f"Ingestion failed: {e}")

        # Rebuild BM25 after new document
        new_chunks = app.state.metadata.get_chunks_by_doc(summary["doc_id"])
        app.state.all_chunks.extend(new_chunks)
        app.state.bm25 = BM25Retriever(app.state.all_chunks)

    return summary


@app.post("/reconcile", summary="Re-run citation resolution for all stubs")
def reconcile():
    '''
    For every stub (:Paper) node, check whether it can now be matched to
    an ingested paper (e.g. a paper was ingested after the stub was created).
    Useful as a periodic cron job as the corpus grows.
    '''
    stubs   = app.state.metadata.list_documents(status="stub")
    updated = 0
    resolver = CitationResolver(app.state.metadata)
    for stub in stubs:
        # Re-attempt matching with current Neo4j state
        ref = RawReference(
            title    = stub.get("title", ""),
            authors  = stub.get("authors", []),
            year     = stub.get("year"),
            doi      = stub.get("doi", ""),
            cite_key = stub.get("doc_id"),
        )
        doc_found, confidence, tier = resolver._fuzzy_title_match(ref.title)
        if doc_found and doc_found != stub["doc_id"] and confidence >= 0.85:
            updated += 1

    return {"stubs_checked": len(stubs), "potentially_resolvable": updated}


print("✅ FastAPI app defined")
print("   GET  /                              health + stats")
print("   GET  /search                        hybrid search")
print("   GET  /documents                     list all papers")
print("   GET  /document/{id}                 single paper record")
print("   GET  /document/{id}/citations       outgoing edges + confidence")
print("   GET  /document/{id}/cited_by        incoming edges")
print("   POST /ingest                        upload PDF + .bib")
print("   POST /reconcile                     resolve stubs")

In [ ]:
# ── Start server in background thread ────────────────────────────────────────
API_PORT = 8765

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=API_PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

print(f"🚀 API running  → http://localhost:{API_PORT}")
print(f"📖 Swagger UI   → http://localhost:{API_PORT}/docs")

## Section 12 — End-to-End Demo

Let's exercise the full API — search, citation graph, and document inspection.

In [ ]:
import httpx, json
API_PORT = "22922"
BASE = f"http://localhost:{API_PORT}"

# ── Health ────────────────────────────────────────────────────────────────────
resp = httpx.get(f"{BASE}/")
print("Health:", json.dumps(resp.json(), indent=2))

In [ ]:
# ── Hybrid search ────────────────────────────────────────────────────────────
def pretty_search(query: str, mode: str = "hybrid", top_k: int = 5,
                  chunk_type: str = None):
    params = {"q": query, "mode": mode, "top_k": top_k}
    if chunk_type:
        params["chunk_type"] = chunk_type

    resp = httpx.get(f"{BASE}/search", params=params)
    data = resp.json()

    md = f"### 🔎 `{mode.upper()}` — *{query}*\n\n"
    for i, r in enumerate(data["results"], 1):
        score = r.get("rrf_score", r.get("score", "?"))
        p     = r["payload"]
        md += (
            f"**[{i}]** `score={score}` | "
            f"📄 `{p['doc_id']}` p.{p['page_num']} | "
            f"🏷 `{p['chunk_type']}` | 📌 *{p.get('section','')}*\n\n"
            f"> {p['text'][:280]}...\n\n---\n"
        )
    display(Markdown(md))

pretty_search("how does multi-head attention work")

In [ ]:
# ── Abstract-only search ─────────────────────────────────────────────────────
pretty_search("transformer architecture for sequence transduction",
              chunk_type="abstract")

In [ ]:
# ── Citation graph via API ───────────────────────────────────────────────────
resp = httpx.get(f"{BASE}/document/vaswani2017attention/citations",
                 params={"min_confidence": 0.0})
data = resp.json()

print(f"📤 Citations from vaswani2017attention  ({data['citation_count']} total)")
print(f"{'─'*75}")
print(f"{'doc_id':<35} {'conf':>5}  {'tier':<28} {'title'}")
print(f"{'─'*75}")
for e in sorted(data["citations"], key=lambda x: -x["confidence"])[:12]:
    print(
        f"{e['doc_id']:<35} {e['confidence']:>5.2f}  "
        f"{e['match_tier']:<28} {e['cited_title'][:35]}"
    )

In [ ]:
# ── Filter by confidence ─────────────────────────────────────────────────────
resp = httpx.get(f"{BASE}/document/vaswani2017attention/citations",
                 params={"min_confidence": 0.8})
data = resp.json()
print(f"📤 Citations with confidence ≥ 0.8: {data['citation_count']}")
for e in data["citations"]:
    print(f"  {e['doc_id']:<35} {e['confidence']:.2f}  {e['match_tier']}")

In [ ]:
# ── Confidence distribution ──────────────────────────────────────────────────
resp   = httpx.get(f"{BASE}/document/vaswani2017attention/citations",
                   params={"min_confidence": 0.0})
confs  = [e["confidence"] for e in resp.json()["citations"]]
tiers  = [e["match_tier"]  for e in resp.json()["citations"]]

from collections import Counter
tier_counts = Counter(tiers)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(confs, bins=[0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.01],
         color="steelblue", edgecolor="white", rwidth=0.85)
ax1.set_xlabel("Confidence Score"); ax1.set_ylabel("Count")
ax1.set_title("Citation Confidence Distribution")

ax2.barh(list(tier_counts.keys()), list(tier_counts.values()), color="coral")
ax2.set_xlabel("Count"); ax2.set_title("Match Tier Distribution")

plt.tight_layout(); plt.show()

print(f"\nMatch tier breakdown:")
for tier, count in sorted(tier_counts.items(), key=lambda x: -x[1]):
    print(f"  {tier:<30}: {count}")

### 📊 Bonus — Cypher queries you couldn't easily do in Week 5

Now that citations are first-class graph relationships, queries that were awkward array-traversals in MongoDB become natural Cypher patterns. Here are a few — run them directly against the Neo4j driver to see the graph at work.

In [ ]:
# ── Direct Cypher: the citation graph as a graph ─────────────────────────────
# These are queries that would be painful with MongoDB array fields but
# are one-liners now.

def run_cypher(query: str, **params):
    '''Helper to run a Cypher query and return rows as dicts.'''
    with metadata_store.driver.session() as s:
        return [dict(r) for r in s.run(query, **params)]

# 1. Most-cited papers within the corpus (incoming :CITES count)
print("🏆 Top 10 most-cited papers in the corpus:")
print("─" * 70)
rows = run_cypher("""
    MATCH (cited:Paper)<-[:CITES]-(citing:Paper)
    RETURN cited.doc_id   AS doc_id,
           cited.title    AS title,
           cited.status   AS status,
           count(citing)  AS times_cited
    ORDER BY times_cited DESC
    LIMIT 10
""")
for r in rows:
    title = (r["title"] or "")[:50]
    print(f"  {r['times_cited']:>3}× | {r['status']:<9} | {r['doc_id']:<30} {title}")

In [ ]:
# 2. Co-citation: papers that share at least one cited paper with vaswani
print("\n🤝 Papers sharing cited references with vaswani2017attention:")
print("─" * 70)
rows = run_cypher("""
    MATCH (x:Paper {doc_id: $doc_id})-[:CITES]->(shared:Paper)<-[:CITES]-(other:Paper)
    WHERE other <> x
    RETURN other.doc_id      AS doc_id,
           other.title       AS title,
           count(shared)     AS shared_refs
    ORDER BY shared_refs DESC
    LIMIT 10
""", doc_id="vaswani2017attention")
if not rows:
    print("  (no co-citations yet — need ≥ 2 ingested papers with overlapping refs)")
for r in rows:
    title = (r["title"] or "")[:50]
    print(f"  {r['shared_refs']:>3} shared | {r['doc_id']:<30} {title}")

In [ ]:
# 3. Two-hop citation reach: papers cited by the papers vaswani cites
print("\n🔭 Two-hop citation reach from vaswani2017attention:")
print("─" * 70)
rows = run_cypher("""
    MATCH (x:Paper {doc_id: $doc_id})-[:CITES]->(mid:Paper)-[:CITES]->(far:Paper)
    WHERE far <> x
    RETURN DISTINCT far.doc_id AS doc_id,
                    far.title  AS title,
                    far.status AS status
    LIMIT 15
""", doc_id="vaswani2017attention")
if not rows:
    print("  (no two-hop targets — Vaswani's direct citations are mostly stubs without onward edges yet)")
for r in rows:
    title = (r["title"] or "")[:50]
    print(f"  {r['status']:<9} | {r['doc_id']:<30} {title}")

In [ ]:
# 4. Confidence-filtered traversal — only "trusted" edges
print("\n✅ High-confidence citation neighbourhood (conf >= 0.85):")
print("─" * 70)
rows = run_cypher("""
    MATCH (x:Paper {doc_id: $doc_id})-[c:CITES]->(cited:Paper)
    WHERE c.confidence >= 0.85
    RETURN cited.doc_id   AS doc_id,
           cited.title    AS title,
           c.confidence   AS confidence,
           c.match_tier   AS match_tier
    ORDER BY c.confidence DESC
""", doc_id="vaswani2017attention")
for r in rows[:10]:
    title = (r["title"] or "")[:40]
    print(f"  {r['confidence']:.2f} {r['match_tier']:<25} | {r['doc_id']:<30} {title}")

### 👥 Bonus — Author + co-authorship queries

Now we exercise the author side of the graph. These queries traverse the `[:WROTE]` and `[:CO_AUTHORED_WITH]` relationships we built during ingestion.

In [ ]:
# 5. Author productivity — top authors by paper count in the corpus
print("🖋️  Top authors by paper count:")
print("─" * 70)
rows = run_cypher("""
    MATCH (a:Author)-[:WROTE]->(p:Paper)
    RETURN a.name        AS author,
           a.name_norm   AS name_norm,
           count(p)      AS papers
    ORDER BY papers DESC
    LIMIT 10
""")
for r in rows:
    print(f"  {r['papers']:>3} papers | {r['author']:<35} ({r['name_norm']})")

In [ ]:
# 6. First-author papers vs. all papers — using the order property on :WROTE
print("\n👑 First-author papers per author:")
print("─" * 70)
rows = run_cypher("""
    MATCH (a:Author)-[w:WROTE]->(p:Paper {status: 'ingested'})
    WHERE w.order = 1
    RETURN a.name AS author, collect(p.doc_id) AS first_author_papers
    ORDER BY size(first_author_papers) DESC
    LIMIT 10
""")
for r in rows:
    papers = ", ".join(r["first_author_papers"][:3])
    if len(r["first_author_papers"]) > 3:
        papers += f" + {len(r['first_author_papers'])-3} more"
    print(f"  {r['author']:<30} : {papers}")

In [ ]:
# 7. Top co-authorship pairs in the corpus
print("\n🤝 Top co-author pairs (most papers in common):")
print("─" * 70)
rows = run_cypher("""
    MATCH (a:Author)-[c:CO_AUTHORED_WITH]->(b:Author)
    RETURN a.name        AS author_a,
           b.name        AS author_b,
           c.paper_count AS papers
    ORDER BY papers DESC
    LIMIT 10
""")
if not rows:
    print("  (no co-authorship edges — need ingested papers with overlapping authors)")
for r in rows:
    print(f"  {r['papers']:>2}× shared | {r['author_a']} ↔ {r['author_b']}")

In [ ]:
# 8. Composing both graphs: who do Vaswani's collaborators cite most?
print("\n🔗 Papers cited by collaborators of 'ashish vaswani':")
print("─" * 70)
rows = run_cypher("""
    MATCH (me:Author {name_norm: $me})-[:CO_AUTHORED_WITH]-(co:Author)
          -[:WROTE]->(:Paper)-[:CITES]->(target:Paper)
    RETURN target.doc_id      AS doc_id,
           target.title       AS title,
           count(*)           AS times
    ORDER BY times DESC
    LIMIT 10
""", me="ashish vaswani")
if not rows:
    print("  (no results — need more ingested papers with overlapping co-authors)")
for r in rows:
    title = (r["title"] or "")[:50]
    print(f"  {r['times']:>2}× | {r['doc_id']:<30} {title}")

## Section 13 — Exercises 🏋️

These exercises are designed to deepen your understanding of each pipeline stage. Work through them in order — each builds on the previous.

---

### Exercise 1 — Layout Sensitivity Test

Run the same PDF through both `layout_mode="text"` and `layout_mode="blocks"` and compare the output quality.

**Task:** For 3 pages of the Vaswani paper:
1. Extract text with each mode
2. Compute the number of sentences that appear "broken" (contain a mid-sentence column jump, detectable by a capital letter appearing after a lowercase word without punctuation)
3. Print a side-by-side comparison

**Goal:** Understand when to switch layout modes.

In [ ]:
# Exercise 1: Layout mode comparison

def count_broken_sentences(text: str) -> int:
    '''
    Heuristic: count occurrences of lowercase word immediately
    followed by a capitalised word without punctuation between them.
    e.g. "sequence The decoder" suggests column bleed.
    '''
    # TODO: implement
    raise NotImplementedError("Your turn! 🚧")

extractor_text   = PDFTextExtractor(layout_mode="text")
extractor_blocks = PDFTextExtractor(layout_mode="blocks")

pages_text   = extractor_text.extract_pages(PDF_PATH,   doc_id="test")
pages_blocks = extractor_blocks.extract_pages(PDF_PATH, doc_id="test")

print("Page | text-mode broken | blocks-mode broken")
print("─" * 45)
for pt, pb in zip(pages_text[:5], pages_blocks[:5]):
    # bt = count_broken_sentences(pt["text"])
    # bb = count_broken_sentences(pb["text"])
    # print(f"  {pt['page_num']:2d} |       {bt:3d}        |       {bb:3d}")
    pass  # uncomment above when implemented

---

### Exercise 2 — Ingest BERT and explore cross-paper citations

1. Download the BERT paper PDF from `https://arxiv.org/pdf/1810.04805`
2. Create a matching `devlin2018bert.bib` with the correct bibliographic metadata
3. Ingest both files using `POST /ingest`
4. Search for `"masked language model"` — do results come from both papers?
5. Check whether any of Vaswani's stubs get resolved when BERT is ingested
6. Query `/document/devlin2018bert/citations` and compare confidence scores with Vaswani

**Goal:** Understand how the citation graph evolves as the corpus grows.

In [ ]:
# Exercise 2: Ingest BERT paper

BERT_BIB = '''
@inproceedings{devlin2018bert,
  author    = {Jacob Devlin and Ming-Wei Chang and Kenton Lee and Kristina Toutanova},
  title     = {{BERT}: Pre-training of Deep Bidirectional Transformers for Language Understanding},
  booktitle = {Proceedings of NAACL-HLT},
  year      = {2019},
  doi       = {10.18653/v1/N19-1423},
  keywords  = {BERT, pre-training, language model, transformers}
}
'''

# Step 1: Write BERT .bib
# Path("devlin2018bert.bib").write_text(BERT_BIB)

# Step 2: Download BERT PDF
# BERT_PDF = "devlin2018bert.pdf"
# if not Path(BERT_PDF).exists():
#     urllib.request.urlretrieve("https://arxiv.org/pdf/1810.04805", BERT_PDF)

# Step 3: Ingest via API
# with open(BERT_PDF, "rb") as pdf_f, open("devlin2018bert.bib", "rb") as bib_f:
#     resp = httpx.post(
#         f"{BASE}/ingest",
#         files={"pdf": ("devlin2018bert.pdf", pdf_f, "application/pdf"),
#                "bib": ("devlin2018bert.bib", bib_f, "text/plain")},
#         timeout=120,
#     )
#     print(resp.json())

# Step 4: Check stubs resolved
# stubs_after = httpx.get(f"{BASE}/documents", params={"status": "stub"}).json()
# print(f"Stubs remaining: {stubs_after['count']}")

# Step 5: Cross-paper search
# pretty_search("masked language model pretraining", mode="hybrid")

---

### Exercise 3 — Abstract Chunk Boosting

The abstract chunk is semantically dense — it summarises the paper's contribution in ~150 words. Implement **score boosting** for abstract chunks in search results.

**Task:** 
1. After retrieving hybrid results, apply a `boost_factor=1.2` multiplier to the `rrf_score` of any chunk where `chunk_type=="abstract"`
2. Re-sort the results
3. Compare top-5 results with and without boosting for the query `"attention mechanism for machine translation"`

**Discussion question:** When might boosting abstracts be *harmful*? (Hint: think about specific technical queries vs. topic-discovery queries.)

In [ ]:
# Exercise 3: Abstract chunk boosting

def search_with_boost(
    query:        str,
    boost_factor: float = 1.2,
    top_k:        int   = 5,
) -> List[Dict]:
    '''
    Hybrid search with score boost for abstract chunks.
    '''
    qvec   = embedder.embed_query(query).tolist()
    dense  = vector_store.search(qvec, top_k=top_k * 2)
    sparse = bm25_retriever.search(query, top_k=top_k * 2)
    hybrid = reciprocal_rank_fusion(dense, sparse, top_k=top_k * 4)

    # TODO: apply boost_factor to abstract chunks, re-sort, return top_k
    raise NotImplementedError("Your turn! 🚧")


# Test:
# results_normal  = reciprocal_rank_fusion(
#     vector_store.search(embedder.embed_query("attention mechanism").tolist(), top_k=10),
#     bm25_retriever.search("attention mechanism", top_k=10), top_k=5
# )
# results_boosted = search_with_boost("attention mechanism", boost_factor=1.2)

---

### Exercise 4 — Citation-Augmented Retrieval (now graph-native!)

Standard RAG retrieves chunks by semantic similarity to the query. A more powerful approach uses the **citation graph** to expand the result set: if chunk C is relevant, papers that cite C's paper (or are cited by it) are likely also relevant.

**With Neo4j this gets much cleaner than the Week 5 version.** You can express the graph expansion as a single Cypher traversal instead of multiple MongoDB lookups.

**Task:** Implement a `citation_augmented_search` function:
1. Run standard hybrid search → top-5 chunks
2. For each result's `doc_id`, fetch the **N-hop citation neighbourhood** in Cypher (try `*1..2` for 1- and 2-hop neighbours)
3. For each neighbour paper, fetch its abstract chunk from Qdrant
4. Score the abstract chunks with the query and add them to results (with a `source="citation_graph"` tag)
5. Re-rank the combined set and return top-10

**Goal:** See if citation-augmented retrieval surfaces relevant papers that raw semantic search missed.

**Cypher hint:**
```cypher
MATCH (start:Paper {doc_id: $doc_id})-[:CITES*1..2]-(neighbour:Paper)
WHERE neighbour.status = 'ingested'
RETURN DISTINCT neighbour.doc_id
```
The undirected `-[:CITES*1..2]-` matches both directions — papers `start` cites AND papers that cite `start`, up to 2 hops away.

In [ ]:
# Exercise 4: Citation-augmented retrieval (Neo4j edition)

def citation_augmented_search(
    query:          str,
    top_k:          int   = 10,
    min_confidence: float = 0.7,
    hops:           int   = 2,
) -> List[Dict]:
    '''
    Hybrid search expanded with citation graph neighbours.

    Steps:
        1. Standard hybrid search (top 5)
        2. For each result's doc_id, traverse :CITES *1..hops to gather neighbours
           (using min_confidence as an edge filter)
        3. Fetch abstract chunks for those neighbours from Qdrant
        4. Add to candidate pool with source="citation_graph"
        5. Re-rank by RRF and return top_k

    Cypher sketch:
        MATCH (s:Paper {doc_id: $doc_id})-[c:CITES*1..$hops]-(n:Paper)
        WHERE ALL(rel IN c WHERE rel.confidence >= $min_conf)
          AND n.status = 'ingested'
        RETURN DISTINCT n.doc_id
    '''
    # TODO: implement
    raise NotImplementedError("Your turn! 🚧")


# Test after ingesting ≥ 2 papers:
# results = citation_augmented_search("sequence to sequence models", top_k=10)
# for r in results:
#     print(f"  source={r.get('source','search'):<16} | {r['payload']['text'][:80]}...")

---

### Exercise 5 (Advanced) — Cross-Encoder Re-ranking

The bi-encoder (BGE-small) scores query and document *independently*, making it fast but approximate. A **cross-encoder** reads query and chunk *together*, producing a much more accurate relevance score — at the cost of higher latency.

Implement a two-stage pipeline:
1. **Stage 1 (recall):** Retrieve top-20 with hybrid search
2. **Stage 2 (precision):** Re-rank with `cross-encoder/ms-marco-MiniLM-L-6-v2`
3. Return top-5

Measure and compare latency for 5 queries:

| Query | Hybrid top-1 | + Cross-encoder top-1 | Latency (ms) |
|---|---|---|---|
| … | … | … | … |

**Discussion:** At what corpus size does the cross-encoder latency become unacceptable? What's the architectural solution? (Hint: the answer involves the two stages you just implemented.)

In [ ]:
# Exercise 5: Cross-encoder re-ranking
# !pip install -q sentence-transformers  (already installed)

# from sentence_transformers import CrossEncoder

# reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# def rerank(query: str, candidates: List[Dict], top_k: int = 5) -> List[Dict]:
#     '''Re-rank hybrid search results with a cross-encoder.'''
#     pairs  = [(query, c["payload"]["text"]) for c in candidates]
#     scores = reranker.predict(pairs)
#     for c, s in zip(candidates, scores):
#         c["rerank_score"] = float(s)
#     return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

# import time
# TEST_QUERIES = [
#     "multi-head attention mechanism",
#     "positional encoding sinusoidal",
#     "encoder decoder architecture",
#     "feed-forward network transformer",
#     "layer normalisation residual connection",
# ]
# for q in TEST_QUERIES:
#     t0 = time.time()
#     hyb = reciprocal_rank_fusion(
#         vector_store.search(embedder.embed_query(q).tolist(), top_k=20),
#         bm25_retriever.search(q, top_k=20), top_k=20
#     )
#     t1 = time.time()
#     top = rerank(q, hyb, top_k=5)
#     t2 = time.time()
#     print(f"  hybrid={1000*(t1-t0):.0f}ms  rerank={1000*(t2-t1):.0f}ms  | {q}")

---

### Exercise 6 (New, Neo4j-specific) — Graph-only queries

These queries are interesting precisely *because* they require graph traversal — they would be slow or awkward with MongoDB arrays.

**Task:** Write Cypher queries (using `run_cypher(...)` defined earlier) that answer:

1. **Most-influential bridge papers** — papers cited by many *ingested* papers in the corpus (not just stubs). Sort by in-degree.
2. **Citation chains** — find the longest directed `[:CITES]` path between any two papers in the corpus. Hint: `MATCH p = (a:Paper)-[:CITES*]->(b:Paper) RETURN p ORDER BY length(p) DESC LIMIT 1`.
3. **Disconnected components** — papers that have no incoming AND no outgoing edges. (These suggest a problem with reference extraction.)
4. **Mutual citations** — pairs of papers that cite each other (rare in the academic literature, but a useful sanity check).
5. **Edge-property analysis** — average confidence per `match_tier`. Does fuzzy matching really land where we expect?

**Goal:** Get comfortable thinking *in graph patterns*. Each query above is 2–4 lines of Cypher.

In [ ]:
# Exercise 6: Graph-only queries

# 1. Most-influential bridge papers (cited by ingested papers)
# rows = run_cypher("""
#     MATCH (cited:Paper)<-[:CITES]-(citing:Paper {status: 'ingested'})
#     RETURN cited.doc_id AS doc_id, count(DISTINCT citing) AS n_citing
#     ORDER BY n_citing DESC LIMIT 10
# """)

# 2. Longest citation chain
# rows = run_cypher("""
#     MATCH path = (a:Paper)-[:CITES*]->(b:Paper)
#     RETURN [n IN nodes(path) | n.doc_id] AS chain, length(path) AS hops
#     ORDER BY hops DESC LIMIT 1
# """)

# 3. Disconnected components
# rows = run_cypher("""
#     MATCH (p:Paper)
#     WHERE NOT (p)-[:CITES]-()
#     RETURN p.doc_id, p.title
# """)

# 4. Mutual citations
# rows = run_cypher("""
#     MATCH (a:Paper)-[:CITES]->(b:Paper)-[:CITES]->(a)
#     WHERE id(a) < id(b)
#     RETURN a.doc_id, b.doc_id
# """)

# 5. Average confidence per match tier
# rows = run_cypher("""
#     MATCH ()-[c:CITES]->()
#     RETURN c.match_tier AS tier,
#            avg(c.confidence) AS mean_conf,
#            count(c) AS n
#     ORDER BY n DESC
# """)

# TODO: pick one or two of these to start, then write your own.
raise NotImplementedError("Your turn! 🚧")

---

### Exercise 7 (New, author-graph) — Author analytics across `[:WROTE]` + `[:CITES]`

The author graph composes beautifully with the citation graph. Many interesting questions are 2- or 3-hop patterns that combine `[:WROTE]` and `[:CITES]`.

**Task:** Write Cypher queries (using `run_cypher(...)` defined earlier) that answer:

1. **Author productivity** — list the top 10 authors by number of ingested papers in the corpus.
2. **Most-cited authors** — sum up the citations received by an author's papers; rank authors by total incoming citations.
3. **"Which authors do I cite most?"** — given an author name, find the authors they cite most often (2-hop across `[:WROTE]` + `[:CITES]`).
4. **Top collaborators of an author** — list co-authors of a given author ranked by `paper_count` on the `[:CO_AUTHORED_WITH]` edge.
5. **Erdős-style distance** — find the shortest co-authorship path between two authors, returning the chain of intermediate authors.
6. **Self-citations** — count, per author, how many of their citations target their own papers (a useful sanity check for academic-integrity work).
7. **Citation between collaborators** — for a given author, find papers cited by their direct co-authors (papers their collaborators thought were worth reading).

**Goal:** Get comfortable composing the two graphs (authorship + citation) in a single Cypher pattern. Each query is 3–6 lines.

**Hint for self-citation (query 6):**
```cypher
MATCH (a:Author)-[:WROTE]->(p1:Paper)-[:CITES]->(p2:Paper)<-[:WROTE]-(a)
RETURN a.name, count(*) AS self_citations
ORDER BY self_citations DESC
```
The same author appears twice in the pattern — that's the self-loop.

In [ ]:
# Exercise 7: Author analytics across [:WROTE] + [:CITES]

# 1. Author productivity (top 10 by ingested-paper count)
# rows = run_cypher("""
#     MATCH (a:Author)-[:WROTE]->(p:Paper {status: 'ingested'})
#     RETURN a.name AS author, count(p) AS papers
#     ORDER BY papers DESC LIMIT 10
# """)

# 2. Most-cited authors (sum of incoming citations across their papers)
# rows = run_cypher("""
#     MATCH (a:Author)-[:WROTE]->(p:Paper)<-[:CITES]-(:Paper)
#     RETURN a.name AS author, count(*) AS total_citations
#     ORDER BY total_citations DESC LIMIT 10
# """)

# 3. "Which authors do I cite most?"
# rows = run_cypher("""
#     MATCH (me:Author {name_norm: $me})-[:WROTE]->(:Paper)
#           -[:CITES]->(:Paper)<-[:WROTE]-(other:Author)
#     WHERE other <> me
#     RETURN other.name AS author, count(*) AS cited_count
#     ORDER BY cited_count DESC LIMIT 10
# """, me="ashish vaswani")

# 4. Top collaborators of a given author
# rows = run_cypher("""
#     MATCH (me:Author {name_norm: $me})-[c:CO_AUTHORED_WITH]-(other:Author)
#     RETURN other.name AS collaborator, c.paper_count AS papers_shared
#     ORDER BY papers_shared DESC LIMIT 10
# """, me="ashish vaswani")

# 5. Erdős-style distance — shortest co-authorship path
# rows = run_cypher("""
#     MATCH path = shortestPath(
#         (a:Author {name_norm: $from})-[:CO_AUTHORED_WITH*..6]-(b:Author {name_norm: $to})
#     )
#     RETURN [n IN nodes(path) | n.name] AS chain, length(path) AS distance
# """, **{"from": "ashish vaswani", "to": "jacob devlin"})

# 6. Self-citations
# rows = run_cypher("""
#     MATCH (a:Author)-[:WROTE]->(p1:Paper)-[:CITES]->(p2:Paper)<-[:WROTE]-(a)
#     RETURN a.name AS author, count(*) AS self_citations
#     ORDER BY self_citations DESC LIMIT 10
# """)

# 7. Citation between collaborators
# rows = run_cypher("""
#     MATCH (me:Author {name_norm: $me})-[:CO_AUTHORED_WITH]-(co:Author)
#           -[:WROTE]->(p:Paper)-[:CITES]->(target:Paper)
#     RETURN target.doc_id AS doc_id, target.title AS title, count(*) AS times
#     ORDER BY times DESC LIMIT 10
# """, me="ashish vaswani")

# TODO: pick one or two to start with, then write your own.
raise NotImplementedError("Your turn! 🚧")

---

## Summary

### What we built (Week 6 — Neo4j edition with author graph)

| Component | Tool | Role |
|---|---|---|
| Metadata | `.bib` + `BibParser` | Ground-truth title, authors, year, DOI |
| Reference extraction | `ReferenceExtractor` | 3-tier: embedded `.bib` → `.bbl` → regex |
| Citation resolution | `CitationResolver` | 4-tier matching + confidence scoring |
| **Paper registry** | **Neo4j `(:Paper)` nodes** | **Bibliographic metadata + status** |
| **Author registry** | **Neo4j `(:Author)` nodes** | **De-duplicated by normalised name** |
| **Authorship** | **`(:Author)-[:WROTE {order}]->(:Paper)`** | **Preserves authorship position** |
| **Citation graph** | **`(:Paper)-[:CITES {confidence, match_tier}]->(:Paper)`** | **First-class relationships** |
| **Co-authorship** | **`(:Author)-[:CO_AUTHORED_WITH {paper_count}]-(:Author)`** | **Derived from `[:WROTE]`** |
| **Stub documents** | **Neo4j `(:Paper {status:"stub"})`** | **Cited papers not yet ingested** |
| Chunk metadata | MongoDB (mongomock) | Flexible queries on chunk fields |
| Text extraction | PyMuPDF `get_text` | Body text, heading detection |
| Chunking | Sliding window 400w/50w | Abstract chunk + body chunks |
| Embedding | BGE-small-en-v1.5 384D | Normalised, query-instruction prefix |
| Vector search | Qdrant in-memory | Dense ANN with payload filters |
| Sparse search | BM25 (rank_bm25) | Keyword retrieval |
| Hybrid fusion | RRF k=60 | Score-agnostic rank combination |
| API | FastAPI + uvicorn | Search, citations, ingestion |

### What changed vs. Week 5

| Concern | Week 5 (MongoDB) | Week 6 (Neo4j) |
|---|---|---|
| Modelling citations | `cites[]` array + manual `cited_by[]` write-back | `[:CITES {confidence, match_tier}]` edges |
| Modelling authors | Flat list of strings on the document | `(:Author)` nodes + `[:WROTE {order}]` edges, de-duplicated |
| Modelling co-authorship | Not modelled at all | `(:Author)-[:CO_AUTHORED_WITH {paper_count}]-(:Author)`, derived |
| Get outgoing citations | `documents.find_one(...).cites` | Cypher `MATCH (p)-[:CITES]->(cited)` |
| Get incoming citations | `documents.find_one(...).cited_by` | Cypher `MATCH (citing)-[:CITES]->(p)` |
| Stub → ingested upgrade | Manual preservation of `cited_by` array | Automatic — edges live on the graph |
| Co-citation queries | Multi-step aggregation pipeline | One-line Cypher pattern |
| N-hop citation reach | Iterative joins in app code | `[:CITES*1..N]` in Cypher |
| Confidence filtering on traversal | Post-filter in Python | `WHERE c.confidence >= $min` in Cypher |
| "Authors I cite most" | Impractical (multi-doc join + dedup) | 2-hop Cypher across `[:WROTE]` + `[:CITES]` |
| Co-authorship distance | Impossible without an external graph library | `shortestPath` in Cypher |

### Production upgrade checklist

- [ ] Replace `mongomock` (for chunks) → `MongoClient("mongodb://...")`
- [ ] Already done: connecting to a real Neo4j Aura instance — you're already in production-grade graph storage
- [ ] Replace in-memory Qdrant → `QdrantClient(host="qdrant", port=6333)`  
- [ ] Add a Neo4j full-text index on `:Paper(title)` for production-scale fuzzy matching: `CREATE FULLTEXT INDEX paper_title_ft FOR (p:Paper) ON EACH [p.title]`
- [ ] Upgrade author normalisation — production needs fuzzy author resolution to handle "A. Vaswani" / "Ashish Vaswani" / "Vaswani, A." / "Vaswani, Ashish" robustly
- [ ] Switch co-authorship refresh from full-rebuild to incremental updates if your corpus grows past ~100k authors
- [ ] Add sentence-boundary chunker for higher coherence (Exercise 1 variant)
- [ ] Implement abstract boost (Exercise 3)
- [ ] Implement citation-augmented retrieval using `[:CITES*1..N]` traversal (Exercise 4)
- [ ] Add cross-encoder re-ranker (Exercise 5)
- [ ] Add Celery queue for async ingestion of large batches
- [ ] Add `/reconcile` cron job to resolve stubs as corpus grows
- [ ] Add API authentication (OAuth2 / API keys)
- [ ] Persist Qdrant to disk: `QdrantClient(path="./qdrant_data")`

In [ ]:
# ── Optional: close the Neo4j driver when you're done ────────────────────────
# metadata_store.close()
# print("👋 Neo4j connection closed.")